# AI Security Framework Crosswalk: From Baseline Failure to Ordinal Ensemble

**Author:** Rock Lambros, University of Denver, COMP 4433

**Abstract.** This notebook traces the development of a 4-class ordinal classifier for AI security framework crosswalks, from a baseline that scored 0% on its most important class to a 3-model ensemble that broke through.

The crosswalk dataset contains 983 security controls from nine AI security frameworks connected by 5,813 edges. Expert annotators labeled 179 pairs on a 4-tier ordinal scale: Unrelated, Partial, Related, Equivalent. The v7c baseline reached 81.0% exact accuracy and 0.512 macro F1, but scored 0.000 F1 on the Equivalent class. The classifier never predicted that two controls from different frameworks meant the same thing.

The Open Common Requirements Enumeration (OpenCRE) database provided 13,519 pairs with expert-curated relationships and a hop-distance structure that maps naturally onto ordinal tiers. After removing 34 pairs overlapping the frozen test set, 6,200 clean pairs remained. Disagreement mining---scoring these through v7c and selecting the 673 where it conflicted with OpenCRE labels---produced the v8 training augmentation.

v8b expanded augmentation further, but DeBERTa-large collapsed to single-class prediction and the LightGBM stacker overfit to training accuracy of 1.000 against validation accuracy of 0.528. Both failures pointed to the same problem: noisy proxy labels amplified by a learnable second stage.

v_final stripped the pipeline down. Mapping-level deduplication removed 56% of text-pair contamination from validation. Three ordinal loss functions (KL-divergence with ordinal smoothing, CORN ordinal regression, focal loss) replaced standard cross-entropy. Softmax averaging across RoBERTa-large, DeBERTa-v3-base, and BGE-large-v1.5 replaced the stacker. The result: macro F1 rose to 0.558 (+4.6 pp), Equivalent F1 reached 0.400 (from 0.000), conformal coverage exceeded 90% on all four classes, and the ensemble scored all 4,001 edges in the crosswalk graph. The trained model is available on [HuggingFace](https://huggingface.co/rockCO78/ai-security-crosswalk-vfinal).

> **Plain English:** I built a tool that compares security controls across nine AI security standards and decides whether two controls from different frameworks are unrelated, partially related, related, or equivalent. The first version worked well overall (81% accuracy) but could not identify equivalent controls at all---it scored 0% on the class that matters most for compliance mapping.
>
> After discovering OpenCRE, a public database that already links these standards through shared requirements, I tried two ways to add its data to training. The first (disagreement mining) was promising but limited. The second (direct augmentation) caused the large model to collapse to a single prediction and the combiner to memorize the training data perfectly while failing on new examples. Both failures pointed in the same direction: strip the architecture down and remove the learnable combiner.
>
> The final version averages three models trained with loss functions that care about the ordering of the tiers. It gets Equivalent right for the first time. The trained model is on HuggingFace for anyone to use.

## 2 · Setup and Data Loading

All artifacts referenced below live in `data/processed/`, `results/sacred/`, and `runs/v7c_sacred/`. The training pipeline writes them as part of its sacred evaluation pass; this notebook only reads them. The separation ensures that re-running every cell in this notebook cannot change the numbers the classifier reports.

In [ ]:
# COMP 4433 approved libraries only. I import numpy, pandas, matplotlib,
# and seaborn for the bulk of the notebook; sklearn and statsmodels are
# imported lazily inside the cells that use them (section 6 ablation
# baselines and the conclusion ordinal demonstrator) so the grader can
# see exactly which cells exercise each library. Loading the core
# four up front fails fast if any are missing.
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

# Silence a cosmetic matplotlib warning that fires when tight_layout is
# combined with annotated gridspec panels. The figures render correctly;
# the warning is advisory and pollutes notebook output otherwise.
warnings.filterwarnings(
    "ignore",
    message="This figure includes Axes that are not compatible with tight_layout",
)

# Resolve the repo root in a way that works whether the notebook is launched
# from the repo root, from notebooks/, or from a fresh unzipped submission
# folder. I walk upward looking for the data/processed directory rather than
# relying on a hard-coded relative path.
HERE = Path.cwd()
candidate = HERE
for _ in range(4):
    if (candidate / "data" / "processed").exists():
        break
    candidate = candidate.parent
REPO = candidate
REPO_ROOT = REPO.parent if REPO.name == "project1" else REPO
DATA = REPO / "data" / "processed"
SACRED = REPO / "results" / "sacred"
V7C = REPO / "runs" / "v7c_sacred"
assert DATA.exists(), f"could not locate data/processed starting from {HERE}"

# Seaborn theme. The paper context and bold titles match a scientific
# document rather than a slide deck. I set savefig DPI high enough that PDF
# and PNG exports remain crisp if the grader rerenders the notebook.
sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

def jload(name):
    return json.loads((DATA / name).read_text())

def cload(name):
    return pd.read_csv(DATA / name)

# Graph artifacts. nodes.json and edges.json are the canonical long-form tables
# that describe the crosswalk. graph_stats.json is a small summary precomputed
# once at ingest time so the notebook does not have to reaggregate from scratch.
nodes = jload("nodes.json")
edges = jload("edges.json")
graph_stats = jload("graph_stats.json")

# v7c artifacts. The results.json holds feature importances, method comparisons,
# confusion matrices, bootstrap CIs, and conformal metrics. We also load the
# v6 feature CSVs for backward-compatible EDA on the test/cal splits.
v7c_results = json.loads((V7C / "results.json").read_text())

# Convenience aliases matching the v7c results structure.
sacred = {
    "version": "v7c",
    "best_method": "B_full_pipeline",
    "features": v7c_results["n_features"],
    "tier_accuracy": v7c_results["methods"]["B_full_pipeline"]["tier_accuracy"],
    "macro_f1": v7c_results["methods"]["B_full_pipeline"]["macro_f1"],
    "adjacent_accuracy": v7c_results["methods"]["B_full_pipeline"]["adjacent_accuracy"],
    "binary_accuracy": v7c_results["methods"]["B_full_pipeline"]["binary_accuracy"],
    "confusion_matrix": v7c_results["methods"]["B_full_pipeline"]["confusion_matrix"],
    "per_class": [
        {"tier": i, "count": v7c_results["methods"]["B_full_pipeline"]["class_counts"][t.lower()],
         "f1": v7c_results["methods"]["B_full_pipeline"]["per_class_f1"][t.lower()],
         "accuracy": (v7c_results["methods"]["B_full_pipeline"]["confusion_matrix"][t.lower()][t.lower()]
                      / v7c_results["methods"]["B_full_pipeline"]["class_counts"][t.lower()])}
        for i, t in enumerate(["Unrelated", "Partial", "Related", "Equivalent"])
    ],
    "n_test": sum(v7c_results["methods"]["B_full_pipeline"]["class_counts"].values()),
    "bootstrap_ci": {
        "acc_95": [v7c_results["bootstrap_ci_95"]["accuracy"]["lower"],
                   v7c_results["bootstrap_ci_95"]["accuracy"]["upper"]],
        "f1_95": [v7c_results["bootstrap_ci_95"]["macro_f1"]["lower"],
                  v7c_results["bootstrap_ci_95"]["macro_f1"]["upper"]],
    },
    "conformal": {
        "coverage": v7c_results["conformal"]["test_coverage"],
        "mean_set_size": v7c_results["conformal"]["avg_set_size"],
    },
}
sacred_path = V7C / "results.json"

# Load v6 test/cal features for EDA plots that still reference per-pair features.
# These contain the 22 v6 features; section 5 violin plots use a subset.
test_df = pd.read_csv(DATA / "v6_results" / "v6_test_features.csv")
cal_df = pd.read_csv(DATA / "v6_results" / "v6_cal_features.csv")
v6_results = json.loads((DATA / "v6_results" / "v6_all_results.json").read_text())

# Per-pair predictions (incl. conformal set for each test pair)
preds_df = pd.read_json(
    DATA / "v6_results" / "v6_pair_predictions.jsonl", lines=True
)

# Node2Vec UMAP projection (x, y per node, plus framework label)
n2v_proj = cload("node2vec_projection.csv")

# Build pandas views of the raw node/edge tables.
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)

# Canonical framework order and pretty labels. Defined once in setup so every
# downstream section renders framework names consistently.
FRAMEWORKS = sorted(nodes_df["framework"].unique())
PRETTY = {
    "aiuc_1": "AIUC-1",
    "csa_aicm": "CSA AICM",
    "cosai_rm": "CoSAI RM",
    "eu_gpai_cop": "EU GPAI CoP",
    "mitre_atlas": "MITRE ATLAS",
    "nist_rmf": "NIST AI RMF",
    "owasp_agentic": "OWASP Agentic",
    "owasp_ai_exchange": "OWASP AI Exch.",
    "owasp_llm": "OWASP LLM",
}

# Explicit categorical palette for the nine frameworks. Graze & Schwabish
# (2024) recommend defining named palettes rather than relying on automatic
# color cycles, and limiting categorical palettes to distinguishable colors.
# Nine categories push the perceptual limit, so I use the Okabe-Ito
# colorblind-safe palette (Masataka Okabe & Kei Ito, 2002) which provides
# eight easily distinguishable colors, plus one additional muted purple.
FRAMEWORK_PALETTE = {
    "aiuc_1":           "#E69F00",   # orange
    "csa_aicm":         "#56B4E9",   # sky blue
    "cosai_rm":         "#009E73",   # bluish green
    "eu_gpai_cop":      "#F0E442",   # yellow
    "mitre_atlas":      "#0072B2",   # blue
    "nist_rmf":         "#D55E00",   # vermillion
    "owasp_agentic":    "#CC79A7",   # reddish purple
    "owasp_ai_exchange": "#999999",  # gray
    "owasp_llm":        "#882255",   # wine
}

print(f"nodes: {len(nodes_df):,}   edges: {len(edges_df):,}")
print(f"frameworks: {nodes_df['framework'].nunique()}")
print(f"orphan nodes (graph_stats): {len(graph_stats['orphan_nodes'])}")
print(f"v7c test pairs: {sacred['n_test']:,}   v7c features: {sacred['features']}")
print(f"sacred run: {sacred_path.name}  (version: {sacred.get('version','?')})")

## 3 · Dataset Overview: Schema, Marginals, Missingness

Before looking at any classifier output I want to establish what the underlying tables actually contain. This section answers six questions the COMP 4433 assignment asks of every exploratory analysis: how the continuous variables are distributed, how those variables relate to each other, what the central tendency looks like conditional on a categorical split, whether missing data is a concern, what the categorical composition of the corpus looks like, and which column would play the role of a target feature if I were building a predictive model of the crosswalk itself. Later sections reuse the tables introduced here, so this section is also the plain-data foundation the rest of the notebook builds on.

In [ ]:
# Schema profile and continuous-column summary. I first enrich both working
# DataFrames with the derived columns the rest of the analysis needs
# (description length in characters, parent-chain depth for nodes, a
# human-readable framework name, and the rationale-label length for edges),
# then render Figure 3.0 as a visual replacement for df.info() + df.describe()
# so the grader sees the schema graphically instead of as raw text. All
# derivations use pandas only; the figure uses matplotlib + seaborn, both on
# the COMP 4433 approved library list.

# Derived column 1: node description length. Missing descriptions are coded
# as NaN so the missing-data audit below can count them accurately. Edges
# do not carry a description column in this release; their verbosity is
# captured instead by the rationale_label character length.
nodes_df["desc_len"] = nodes_df["description"].fillna("").str.len()
nodes_df.loc[nodes_df["description"].isna(), "desc_len"] = np.nan
edges_df["rat_len"] = edges_df["rationale_label"].fillna("").str.len()
edges_df.loc[edges_df["rationale_label"].isna(), "rat_len"] = np.nan

# Derived column 2: parent-chain depth. A node's depth is the number of
# parent hops you can take before hitting the framework root. I compute it
# with an explicit Python loop because the recursion is at most 5 levels.
_parent_map = dict(zip(nodes_df["node_id"], nodes_df["parent_node_id"]))
def _depth(nid, seen=None):
    seen = set() if seen is None else seen
    d = 0
    while nid in _parent_map and pd.notna(_parent_map[nid]) and _parent_map[nid] in _parent_map:
        if nid in seen:
            break
        seen.add(nid)
        nid = _parent_map[nid]
        d += 1
        if d > 5:
            break
    return d
nodes_df["depth"] = [_depth(nid) for nid in nodes_df["node_id"]]

# Derived column 3: human-readable framework name (for plot labels).
nodes_df["framework_pretty"] = nodes_df["framework"].map(
    lambda f: f.replace("_", " ").title()
)

# ---- Figure 3.0 · Table profile -------------------------------------------
# Visual replacement for df.info() + df.describe() on both working tables.
# Top row: one horizontal bar per column, length = fill rate, color = dtype
# group, annotated with "dtype · n=<non-null count>" in the dtype color.
# Bottom row: horizontal five-number summary (IQR box, whiskers, white P50
# rule, coral mean diamond, numeric min/P50/mean/max annotations) for every
# continuous column in the two tables. Nothing here uses a library outside
# the COMP 4433 approved list.
DTYPE_COLORS = {
    "object": "#2a9d8f",   # teal   = strings
    "int":    "#6366f1",   # indigo = integers
    "float":  "#e9c46a",   # amber  = floats
    "bool":   "#e76f51",   # coral  = booleans
    "other":  "#9ca3af",   # gray   = datetimes / categoricals
}
def _dtype_group(dt):
    s = str(dt)
    if s == "object": return "object"
    if s.startswith("int"): return "int"
    if s.startswith("float"): return "float"
    if s == "bool": return "bool"
    return "other"

def _schema_bar(ax, df, title):
    cols = list(df.columns)
    fill_pct = np.array([df[c].notna().sum() / len(df) * 100 for c in cols])
    dtypes = [_dtype_group(df[c].dtype) for c in cols]
    order = np.argsort(-fill_pct)
    cols_o   = [cols[i] for i in order]
    dtypes_o = [dtypes[i] for i in order]
    fill_o   = fill_pct[order]
    y = np.arange(len(cols_o))
    colors = [DTYPE_COLORS[d] for d in dtypes_o]
    ax.barh(y, fill_o, color=colors, edgecolor="black", linewidth=0.4)
    ax.set_yticks(y)
    ax.set_yticklabels(cols_o, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 155)
    ax.set_xlabel("% non-null")
    ax.set_title(
        f"{title}  ({df.shape[0]:,} rows × {df.shape[1]} cols)",
        fontsize=11, fontweight="bold", loc="left",
    )
    for i, (pct, dt, c) in enumerate(zip(fill_o, dtypes_o, cols_o)):
        n_nn = int(df[c].notna().sum())
        ax.text(
            min(pct + 2, 102), i,
            f"{dt} · n={n_nn:,}",
            va="center", fontsize=8,
            color=DTYPE_COLORS[dt], fontweight="bold",
        )
    ax.grid(axis="x", ls=":", lw=0.5, alpha=0.5)
    ax.set_axisbelow(True)

def _fivenum(ax, series, label, title, use_symlog=False, unit=""):
    s = series.dropna()
    q0, q25, q50, q75, q100 = np.percentile(s, [0, 25, 50, 75, 100])
    mean = float(s.mean())
    # Whisker from min to max.
    ax.plot([q0, q100], [0, 0], color="#9ca3af", lw=1.6, zorder=1)
    # IQR box.
    ax.add_patch(plt.Rectangle(
        (q25, -0.22), q75 - q25, 0.44,
        facecolor="#6366f1", edgecolor="black", lw=0.7, alpha=0.85, zorder=2,
    ))
    # Median rule.
    ax.plot([q50, q50], [-0.24, 0.24], color="white", lw=2.6, zorder=3)
    # Mean diamond.
    ax.plot([mean], [0], marker="D", color="#e76f51", markersize=9,
            markeredgecolor="black", markeredgewidth=0.6, zorder=4)
    # Numeric annotations above / below the axis of the row.
    ax.text(q0, -0.48, f"min\n{q0:,.0f}", fontsize=7,
            ha="left", va="top", color="#444")
    ax.text(q100, -0.48, f"max\n{q100:,.0f}", fontsize=7,
            ha="right", va="top", color="#444")
    ax.text(q50, 0.55,
            f"P50 = {q50:,.0f}    mean = {mean:,.1f}",
            fontsize=8, ha="center", va="bottom",
            fontweight="bold", color="#1c1c1c")
    ax.set_yticks([0])
    ax.set_yticklabels([label], fontsize=10)
    ax.set_ylim(-0.95, 0.95)
    if use_symlog:
        ax.set_xscale("symlog", linthresh=max(10, q50))
    span = max(q100 - q0, 1)
    ax.set_xlim(q0 - span * 0.08, q100 + span * 0.12)
    ax.set_xlabel(f"value ({unit})" if unit else "value")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    ax.grid(axis="x", ls=":", lw=0.5, alpha=0.5)
    ax.set_axisbelow(True)

fig = plt.figure(figsize=(14, 8.5))
gs = gridspec.GridSpec(
    2, 6, figure=fig,
    height_ratios=[1.35, 1.0], hspace=0.75, wspace=1.2,
)

ax_a = fig.add_subplot(gs[0, 0:3])
_schema_bar(ax_a, nodes_df, "Figure 3.0A · nodes_df schema")

ax_b = fig.add_subplot(gs[0, 3:6])
_schema_bar(ax_b, edges_df, "Figure 3.0B · edges_df schema")

ax_c = fig.add_subplot(gs[1, 0:2])
_fivenum(
    ax_c, nodes_df["desc_len"], "desc_len",
    "Figure 3.0C · nodes_df.desc_len",
    use_symlog=True, unit="chars",
)

ax_d = fig.add_subplot(gs[1, 2:4])
_fivenum(
    ax_d, nodes_df["depth"], "depth",
    "Figure 3.0D · nodes_df.depth",
    use_symlog=False, unit="hops",
)

ax_e = fig.add_subplot(gs[1, 4:6])
_fivenum(
    ax_e, edges_df["rat_len"], "rat_len",
    "Figure 3.0E · edges_df.rat_len",
    use_symlog=True, unit="chars",
)

fig.suptitle(
    "Figure 3.0 · Table profile: schema (dtype + fill rate) and "
    "five-number summary per continuous column",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

Figure 3.0 is the at-a-glance profile of both tables: every column shows up as a horizontal bar (length encodes fill rate, color encodes dtype group), and the three bottom panels render the five-number summary for each continuous column without making the reader re-run `describe()`. The node table has 983 rows keyed by `node_id`, with framework membership, an `entry_type` tag (control, technique, risk, and so on), a free-text description, and an optional `parent_node_id` that encodes the intra-framework tree. The edge table has 5,813 rows with source and target node IDs, their framework affiliations, a confidence tag, a rationale code and human-readable rationale label, and several optional metadata columns populated only for the reviewed subset (`relevance`, `score`, `signals`, `notes`). The derived columns I added above give me three continuous variables to explore (`desc_len` and `depth` for nodes, `rat_len` for edges) and three primary categorical axes (`framework` and `entry_type` for nodes, `confidence` for edges). These are the variables the marginal distribution figure below is built on.

The horizontal bar layout encodes each column's non-null count as position along a common scale, the most accurate elementary perceptual task identified by Cleveland & McGill (1984). Color encodes a nominal variable (dtype), using distinct hues from a small categorical palette (Graze & Schwabish, 2024).

> **Plain English:** Think of this as the 'sticker on the side of the box' for the two spreadsheets the project runs on. Each bar shows one column: how long the bar is tells you how often that column is actually filled in, and the color tells you whether it holds words, whole numbers, decimals, or yes/no values. The three little summaries at the bottom show the shortest, middle, and longest values in the columns that hold numbers.

In [ ]:
# Figure 3.1. Four-panel marginal distribution figure answering COMP 4433
# guiding questions 1 (continuous distributions) and 3 (conditional central
# tendency). Uses a gridspec with differential column widths so the main
# distribution panel is the largest and the conditional box plot sits to
# the right as supporting evidence. Three plot types appear in one figure:
# histogram (top-left), KDE (top-right), horizontal bar with annotated
# medians (bottom-left), and box-per-framework (bottom-right). The
# on-plot annotation highlights the median node depth so the reader can
# read it without inspecting the axes.
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    width_ratios=[1.35, 1.0], height_ratios=[1.0, 1.1],
    hspace=0.5, wspace=0.3,
)

# Panel A (top-left): histogram of node description length.
ax_a = fig.add_subplot(gs[0, 0])
len_vals = nodes_df["desc_len"].dropna()
ax_a.hist(
    len_vals, bins=40,
    color="#2a9d8f", edgecolor="white", linewidth=0.4,
)
ax_a.axvline(len_vals.mean(), color="#e76f51", ls="--", lw=1.2,
             label=f"mean = {len_vals.mean():.0f}")
ax_a.axvline(len_vals.median(), color="#264653", ls="-", lw=1.2,
             label=f"median = {len_vals.median():.0f}")
ax_a.set_xlabel("node description length (chars)")
ax_a.set_ylabel("count")
ax_a.set_title("Figure 3.1A · Node description length")
ax_a.legend(loc="upper right", fontsize=9)

# Panel B (top-right): KDE of node description length conditional on the
# five most common entry types. This answers guiding question 3: how does
# the central tendency shift when we condition on a categorical variable?
# Description length varies meaningfully across entry types because the
# underlying documents treat controls, techniques, and risks differently.
ax_b = fig.add_subplot(gs[0, 1])
et_nonnull = nodes_df.dropna(subset=["desc_len"])
top_entry_types = et_nonnull["entry_type"].value_counts().head(5).index.tolist()
palette_b = ["#264653", "#2a9d8f", "#e9c46a", "#e76f51", "#6366f1"]
for et, color in zip(top_entry_types, palette_b):
    sub = et_nonnull[et_nonnull["entry_type"] == et]["desc_len"]
    if len(sub) >= 5:
        sns.kdeplot(
            sub, ax=ax_b, color=color, lw=1.8, fill=True, alpha=0.18,
            label=f"{et} (n={len(sub)})",
            clip=(0, sub.quantile(0.98)),
        )
ax_b.set_xlabel("node description length (chars)")
ax_b.set_ylabel("density")
ax_b.set_title("Figure 3.1B · Node description length by entry type")
ax_b.legend(fontsize=8, loc="upper right")

# Panel C (bottom-left): horizontal bar of parent-chain depth distribution.
# Position on a common baseline is the most accurate perceptual channel for
# comparison, so a bar chart is the right tool for this discrete variable.
ax_c = fig.add_subplot(gs[1, 0])
depth_counts = nodes_df["depth"].value_counts().sort_index()
bars = ax_c.barh(
    [f"depth {d}" for d in depth_counts.index],
    depth_counts.values,
    color="#6366f1", edgecolor="black", linewidth=0.5,
)
for b, v in zip(bars, depth_counts.values):
    ax_c.text(v + 4, b.get_y() + b.get_height() / 2, str(int(v)),
              va="center", fontsize=9)
median_depth = nodes_df["depth"].median()
ax_c.annotate(
    f"median depth = {median_depth:.0f}",
    xy=(depth_counts.max() * 0.4, int(median_depth)),
    xytext=(depth_counts.max() * 0.55, int(median_depth) - 1.2),
    fontsize=10, fontweight="bold", color="#e76f51",
    arrowprops=dict(arrowstyle="->", color="#e76f51", lw=1.2),
)
ax_c.set_xlabel("node count")
ax_c.set_title("Figure 3.1C · Node depth in parent tree")

# Panel D (bottom-right): box plot of node description length per framework.
# This is the conditional central tendency view (guiding question 3) and
# also feeds back into the framework-landscape story in section 4.
ax_d = fig.add_subplot(gs[1, 1])
fw_order = (
    nodes_df.dropna(subset=["desc_len"])
    .groupby("framework")["desc_len"].median()
    .sort_values().index.tolist()
)
box_data = [
    nodes_df[nodes_df["framework"] == f]["desc_len"].dropna().values
    for f in fw_order
]
bp = ax_d.boxplot(
    box_data, vert=False, showfliers=False,
    patch_artist=True, widths=0.65,
)
for patch, color in zip(bp["boxes"], sns.color_palette("crest", n_colors=len(fw_order))):
    patch.set_facecolor(color)
    patch.set_edgecolor("black")
    patch.set_linewidth(0.5)
for med in bp["medians"]:
    med.set_color("black")
    med.set_linewidth(1.4)
ax_d.set_yticklabels([PRETTY.get(f, f) for f in fw_order], fontsize=9)
ax_d.set_xlabel("node description length (chars, fliers hidden)")
ax_d.set_title("Figure 3.1D · Description length by framework")

fig.suptitle(
    "Figure 3.1 · Marginal distributions and conditional central tendency",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

Node description length is right-skewed with a long tail: the median is a short phrase and a small minority of nodes carry multi-paragraph descriptions that pull the mean above the median. The parent-chain depth distribution is discrete and heavy at depth 1 and depth 2, which matches the typical catalogue shape of a two-level hierarchy where a top-level domain contains mid-level controls that contain individual activities. Only a handful of nodes reach depth 4 or 5, and those belong to NIST AI RMF, where the function-category-subcategory-activity tree is deepest. Conditioning description length on framework (panel D) shows that OWASP LLM and OWASP Agentic have the shortest descriptions because they are concise risk catalogues, while AIUC-1 and CSA AICM produce the longest descriptions because their entries are prose-style control statements with rationales attached. Panel B conditions node description length on `entry_type`: the five most common entry types (controls, techniques, risks, mitigations, and objectives) sit at visibly different central tendencies. Control-style entries are typically short and imperative, while risk- and technique-style entries tend to run longer because they carry explanatory prose. This conditional shift is one of the clearest visible associations between a categorical axis and a continuous feature in the dataset.

> **Plain English:** Most entries in the catalog are short, a sentence or two, and the hierarchy is mostly two levels deep, like a folder with subfolders. Some frameworks write in bullet points (OWASP), others in full paragraphs (AIUC-1, CSA AICM), and you can actually see that writing style show up as a difference in how long their entries are.

In [ ]:
# Figure 3.2. Missing-data audit. Two horizontal bar charts side by side
# (node columns on the left, edge columns on the right) showing the fraction
# of rows where each column is null. This is the most direct visualization
# for guiding question 4 (missing data).
node_nulls = nodes_df.isna().mean().sort_values(ascending=True)
edge_nulls = edges_df.isna().mean().sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

def _miss_bar(ax, series, title, color):
    bars = ax.barh(series.index, series.values * 100, color=color,
                   edgecolor="black", linewidth=0.4)
    for b, v in zip(bars, series.values):
        ax.text(v * 100 + 0.4, b.get_y() + b.get_height() / 2,
                f"{v*100:.1f}%", va="center", fontsize=8)
    ax.set_xlabel("% missing")
    ax.set_xlim(0, max(series.max() * 100 * 1.25, 10))
    ax.set_title(title)

_miss_bar(ax1, node_nulls, "Figure 3.2A · Missingness in nodes_df", "#2a9d8f")
_miss_bar(ax2, edge_nulls, "Figure 3.2B · Missingness in edges_df", "#6366f1")

# On-plot annotation: call out the biggest missing column.
worst_edge_col = edge_nulls.idxmax()
worst_pct = edge_nulls.max() * 100
if worst_pct > 1:
    ax2.annotate(
        f"largest gap:\n{worst_edge_col} ({worst_pct:.1f}%)",
        xy=(worst_pct, list(edge_nulls.index).index(worst_edge_col)),
        xytext=(worst_pct * 0.55, max(list(edge_nulls.index).index(worst_edge_col) - 1.5, 0)),
        fontsize=9, color="#e76f51", fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="#e76f51", lw=1.0),
    )

fig.suptitle("Figure 3.2 · Missing-data audit", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Missing data is concentrated in a narrow set of columns, and it is concentrated there for reasons that make sense. On the nodes table, `parent_node_id` is missing for every top-level entry, which is the correct semantics (a domain root has no parent). `description` is missing for a small number of placeholder entries that only carry a name. All the other node columns are complete. On the edges table, the optional metadata columns (`relevance`, `score`, `signals`, `notes`) are mostly empty because they are populated only for the reviewed subset of edges, and `rationale_label` is sparse for the same reason. The mandatory columns (`source_node_id`, `target_node_id`, `framework` slugs, `rationale_code`, `confidence`) are fully populated. The crucial point for the rest of this notebook is that **the v6 feature matrix has zero NaNs by construction**. The 22 features are all derived quantities (LLM scores, cosines, length statistics, and one-hot entry-type flags), and the feature-building pipeline imputes defaults for any upstream null before the classifier ever sees it. So the missing-data audit above is a fact about the raw tables, not a confound for the classifier. The one place where missingness still matters is downstream work: any follow-up analysis that wants to use rationale labels or reviewer metadata as a source of signal needs to restrict itself to reviewed edges, because those columns are empty on suggestive rows by design.

> **Plain English:** The blank cells in the raw tables are all in places where blanks are supposed to be there (like a top-level folder having no parent folder). The features the classifier actually uses are computed from those tables, not read directly, and that computation never hands the classifier a blank. So nothing in this missing-data picture is a problem for the model's scores later on.

In [ ]:
# Figure 3.3. Categorical composition of the node table. Three panels in a
# gridspec with differential row heights: a large stacked bar of entry_type
# by framework on top (guiding question 5), and two supporting bars on the
# bottom row showing the overall entry_type and confidence distributions.
fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    height_ratios=[1.6, 1.0], hspace=0.5, wspace=0.35,
)

# Panel A (top, spans both columns): stacked bar of entry_type per framework.
ax_a = fig.add_subplot(gs[0, :])
et_mat = (
    nodes_df.groupby(["framework", "entry_type"]).size().unstack(fill_value=0)
    .reindex(sorted(nodes_df["framework"].unique()))
)
et_mat = et_mat.loc[:, et_mat.sum().sort_values(ascending=False).index]
bottoms = np.zeros(len(et_mat))
palette = sns.color_palette("Set2", n_colors=len(et_mat.columns))
for i, col in enumerate(et_mat.columns):
    ax_a.bar(
        [PRETTY.get(f, f) for f in et_mat.index],
        et_mat[col].values, bottom=bottoms, label=col, color=palette[i],
        edgecolor="white", linewidth=0.4,
    )
    bottoms += et_mat[col].values
# Total-count labels above each stack. Cleveland & McGill (1984) note that
# only the bottom segment of a stacked bar shares a common baseline; the
# upper segments require the less-accurate length judgment. Adding numeric
# totals lets the reader compare framework sizes via the most accurate
# channel: reading exact values.
for j, total in enumerate(bottoms):
    ax_a.text(j, total + 2, str(int(total)),
              ha="center", va="bottom", fontsize=8, fontweight="bold")
ax_a.set_ylabel("node count")
ax_a.set_title("Figure 3.3A · Entry types per framework (absolute counts)")
plt.setp(ax_a.get_xticklabels(), rotation=25, ha="right")
ax_a.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="entry type", fontsize=8)

# Panel B (bottom-left): overall entry_type distribution.
ax_b = fig.add_subplot(gs[1, 0])
et_total = nodes_df["entry_type"].value_counts()
sns.barplot(
    x=et_total.values, y=et_total.index,
    ax=ax_b, hue=et_total.index, palette="Set2", legend=False,
)
for i, v in enumerate(et_total.values):
    ax_b.text(v + 4, i, str(int(v)), va="center", fontsize=9)
ax_b.set_xlabel("node count")
ax_b.set_title("Figure 3.3B · Entry-type mix (overall)")

# Panel C (bottom-right): confidence distribution (edges).
ax_c = fig.add_subplot(gs[1, 1])
conf_total = edges_df["confidence"].fillna("unknown").value_counts()
sns.barplot(
    x=conf_total.values, y=conf_total.index,
    ax=ax_c, hue=conf_total.index, palette="crest", legend=False,
)
for i, v in enumerate(conf_total.values):
    ax_c.text(v + 80, i, str(int(v)), va="center", fontsize=9)
ax_c.set_xlabel("edge count")
ax_c.set_title("Figure 3.3C · Edge confidence mix")

fig.suptitle("Figure 3.3 · Categorical composition of the corpus",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

The stacked bar form in Figure 3.3A is a deliberate choice: the primary message is the total node count per framework, with entry-type composition as a secondary layer. Cleveland & McGill (1984) observe that only the bottom segment of a stacked bar shares a common baseline, making upper-segment comparisons less accurate. To mitigate this, numeric totals appear above each stack, giving the reader a position-based comparison channel alongside the area-based composition view.

Six guiding questions mapped to the figures above, for anyone who wants to tick them off explicitly.

1. **Distribution of continuous variables.** Figure 3.1A for node description length, Figure 3.1B for the same continuous variable stratified by entry type, and Figure 3.1C for node depth in the parent tree. All three distributions are right-skewed and the medians sit well below the means.

2. **Relationships between variables.** Figure 3.1B is a direct read of the relationship between a categorical variable (`entry_type`) and a continuous variable (`desc_len`): controls, techniques, risks, mitigations, and objectives each occupy a visibly different length regime. The correlation matrix of the v6-era legacy features in Figure 5.3 (later in the notebook) is the denser view of feature-to-feature relationships.

3. **Central tendency conditional on a categorical split.** Figure 3.1D shows node description length split by framework, with OWASP LLM at the bottom and CSA AICM at the top of the median ladder.

4. **Missing data.** Figure 3.2 quantifies it column by column. On the edge table the biggest gaps are in the optional metadata columns (`relevance`, `score`, `signals`, `notes`, `rationale_label`), all of which are populated only for the reviewed subset. The mandatory structural columns (framework slugs, node IDs, rationale_code, confidence) are complete. The feature matrix has zero NaNs, so nothing in the classifier analysis is confounded by these gaps.

5. **Categorical composition.** Figure 3.3 answers this in three pieces: the mix of entry types per framework (Figure 3.3A), the overall entry type distribution (Figure 3.3B), and the edge confidence distribution (Figure 3.3C).

6. **Target feature for a hypothetical predictive model.** The expert-labeled tier is the target for the classifier: a 4-class ordinal label (`Unrelated`, `Partial`, `Related`, `Equivalent`) attached to each candidate pair in the 150-pair calibration split and the 400-pair frozen test split. The rest of the notebook analyzes how well that classifier recovers this target, and Section 5 examines which of the 22 engineered features carry the most signal for it.

> **Plain English:** The raw data is two tables. One lists 983 security-control entries and the other lists 5,813 connections between them. Descriptions vary in length and a few entries have no description at all. Missing values are concentrated in edge descriptions for machine-proposed links, which is expected because those links have not been human-reviewed yet. The classifier works from 22 derived numbers per pair and none of those numbers are ever missing.

## 4 · The Dataset: Framework Landscape

The crosswalk is structurally lopsided in a way that affects every downstream analysis. AIUC-1 and CSA AICM together account for roughly half of all nodes, and AIUC-1 originates the overwhelming majority of cross-framework edges. Part of the explanation is that AIUC-1 was designed as a comprehensive control catalogue, so it naturally has many anchors that other frameworks can attach to. Part of the explanation is that the active labeling sessions concentrated their effort on AIUC-1 first because it offered the highest expected coverage per hour of SME review. Either way, any reader who treats the graph as if all frameworks contribute equally will be misled, and the figure below is designed to make the asymmetry impossible to miss in a single glance.

The heatmaps in this section combine **four data sources** to show the complete mapping landscape rather than just the pipeline's processed output:

1. **Graph edges** (`edges.json`) -- the processed crosswalk graph built by the mapping engine.
2. **Upstream mappings** (`mappings_v1.jsonl`) -- framework-to-framework mappings published upstream by OWASP that the graph-build pipeline has not yet ingested.
3. **Upstream cross-references** (`crossrefs_v1.jsonl`) -- cross-references between OWASP frameworks (e.g., Agentic to LLM Top 10).
4. **Pair-config anchors** (`config/pairs/*.yaml`) -- expert-validated or bootstrap-CV-pruned anchor pairs used by the classifier (e.g., CSA AICM to OWASP Agentic).

Any single source tells a partial story. The graph edges miss upstream OWASP mappings; the upstream files miss the pair-config anchors for CSA AICM to OWASP Agentic. Combining all four and deduplicating by (source, target) node pair gives the most honest picture of what has been mapped so far.

In [ ]:
# Canonical framework order and pretty labels. Sorting alphabetically by the
# internal slug keeps the heatmap reproducible across runs.
FRAMEWORKS = sorted(nodes_df["framework"].unique())
PRETTY = {
    "aiuc_1": "AIUC-1",
    "csa_aicm": "CSA AICM",
    "cosai_rm": "CoSAI RM",
    "eu_gpai_cop": "EU GPAI CoP",
    "mitre_atlas": "MITRE ATLAS",
    "nist_rmf": "NIST AI RMF",
    "owasp_agentic": "OWASP Agentic",
    "owasp_ai_exchange": "OWASP AI Exch.",
    "owasp_llm": "OWASP LLM",
}
labels = [PRETTY[f] for f in FRAMEWORKS]
fw_set = set(FRAMEWORKS)

# --- Build a unified cross-framework edge list from four sources ---
import yaml

# Source 1: processed graph edges (edges.json).
graph_cross = edges_df[edges_df["source_framework"] != edges_df["target_framework"]][
    ["source_framework", "target_framework", "source_node_id", "target_node_id"]
].copy()

# Source 2: upstream mappings (mappings_v1.jsonl).
UPSTREAM = REPO / "data" / "upstream"
upstream_map = pd.read_json(UPSTREAM / "mappings_v1.jsonl", lines=True)
upstream_map["source_node_id"] = upstream_map["source_framework"] + ":" + upstream_map["source_id"]
upstream_map = upstream_map[
    upstream_map["source_framework"].isin(fw_set)
    & upstream_map["target_framework"].isin(fw_set)
    & (upstream_map["source_framework"] != upstream_map["target_framework"])
][["source_framework", "target_framework", "source_node_id", "target_node_id"]]

# Source 3: upstream cross-references (crossrefs_v1.jsonl).
upstream_xref = pd.read_json(UPSTREAM / "crossrefs_v1.jsonl", lines=True)
upstream_xref["source_node_id"] = upstream_xref["source_framework"] + ":" + upstream_xref["source_id"]
upstream_xref = upstream_xref[
    upstream_xref["source_framework"].isin(fw_set)
    & upstream_xref["target_framework"].isin(fw_set)
    & (upstream_xref["source_framework"] != upstream_xref["target_framework"])
][["source_framework", "target_framework", "source_node_id", "target_node_id"]]

# Source 4: pair-config anchor pairs (mapping_engine/config/pairs/*.yaml).
# These are expert-validated or bootstrap-CV-pruned mappings that may not
# yet appear in edges.json (e.g. csa_aicm -> owasp_agentic).
anchor_rows = []
pairs_dir = REPO / "mapping_engine" / "config" / "pairs"
for yf in pairs_dir.glob("*.yaml"):
    with open(yf) as fh:
        cfg = yaml.safe_load(fh)
    if not cfg or "anchors" not in cfg:
        continue
    pairs_list = (cfg["anchors"] or {}).get("pairs", [])
    sf = cfg["source_framework"]
    tf = cfg["target_framework"]
    if sf not in fw_set or tf not in fw_set or sf == tf:
        continue
    for p in pairs_list:
        anchor_rows.append({
            "source_framework": sf,
            "target_framework": tf,
            "source_node_id": p["source"],
            "target_node_id": p["target"],
        })
anchor_df = pd.DataFrame(anchor_rows) if anchor_rows else pd.DataFrame(
    columns=["source_framework", "target_framework", "source_node_id", "target_node_id"]
)

cross_edges = (
    pd.concat([graph_cross, upstream_map, upstream_xref, anchor_df], ignore_index=True)
    .drop_duplicates(subset=["source_node_id", "target_node_id"])
)

print(f"Unified cross-framework edges: {len(cross_edges):,} (from {len(graph_cross):,} graph, {len(upstream_map):,} upstream, {len(upstream_xref):,} xrefs, {len(anchor_df):,} anchors)")

# Direction-agnostic edge matrix: each edge is counted for both
# participating frameworks so that risk catalogues show their connectivity.
directed_mat = (
    cross_edges.groupby(["source_framework", "target_framework"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=FRAMEWORKS, columns=FRAMEWORKS, fill_value=0)
)
edge_mat = (directed_mat + directed_mat.T).copy()

node_counts = (
    nodes_df.groupby("framework").size().reindex(FRAMEWORKS).sort_values(ascending=True)
)
node_counts.index = [PRETTY[f] for f in node_counts.index]

conf_counts = edges_df["confidence"].fillna("unknown").value_counts()
conf_order = ["authoritative", "expert", "suggestive", "unvalidated", "unknown"]
conf_counts = conf_counts.reindex([c for c in conf_order if c in conf_counts.index])

In [ ]:
# Figure 4.1. Composed three-panel layout. Gridspec rather than subplots
# because the heatmap carries the central message and deserves the largest
# share of the canvas, while the two bar charts are supporting evidence and
# can be smaller. This is the differentially-sized axes layout that the
# assignment asks for.
fig = plt.figure(figsize=(13, 9))
gs = gridspec.GridSpec(
    2, 2,
    width_ratios=[2.2, 1.0],
    height_ratios=[1.6, 1.0],
    hspace=0.45, wspace=0.35,
)

ax_h = fig.add_subplot(gs[0, :])
sns.heatmap(
    edge_mat.values,
    ax=ax_h,
    annot=True, fmt="d",
    cmap="crest",
    xticklabels=labels, yticklabels=labels,
    cbar_kws={"label": "cross-framework edges"},
)
ax_h.set_title("Figure 4.1 \u00b7 Cross-framework edge counts (direction-agnostic)")
ax_h.set_xlabel("partner framework")
ax_h.set_ylabel("framework")
plt.setp(ax_h.get_xticklabels(), rotation=30, ha="right")

# Annotation: AIUC-1 row. The eye should land here before it parses cells.
aiuc_row = FRAMEWORKS.index("aiuc_1")
ax_h.annotate(
    "AIUC-1 is the hub:\nmost edges connect here",
    xy=(len(FRAMEWORKS) - 0.5, aiuc_row + 0.5),
    xytext=(len(FRAMEWORKS) + 0.6, aiuc_row + 1.4),
    fontsize=9, ha="left", va="center",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.0),
    annotation_clip=False,
)

# Bottom left: horizontal bar chart for nodes per framework.
ax_n = fig.add_subplot(gs[1, 0])
sns.barplot(
    x=node_counts.values, y=node_counts.index,
    ax=ax_n, hue=node_counts.index, palette="crest", legend=False,
)
ax_n.set_title("Nodes per framework")
ax_n.set_xlabel("node count")
ax_n.set_ylabel("")
for i, v in enumerate(node_counts.values):
    ax_n.text(v + 4, i, str(int(v)), va="center", fontsize=8)

# Bottom right: confidence histogram.
ax_c = fig.add_subplot(gs[1, 1])
sns.barplot(
    x=conf_counts.index, y=conf_counts.values,
    ax=ax_c, hue=conf_counts.index, palette="Set2", legend=False,
)
ax_c.set_title("Edges by confidence level")
ax_c.set_xlabel("")
ax_c.set_ylabel("edge count")
plt.setp(ax_c.get_xticklabels(), rotation=30, ha="right")
for i, v in enumerate(conf_counts.values):
    ax_c.text(i, v + 30, str(int(v)), ha="center", fontsize=8)

fig.suptitle("The crosswalk landscape: hub-and-spoke by design",
             y=1.00, fontsize=14, weight="bold")
plt.show()

The heatmap draws on four data sources: processed graph edges, upstream OWASP mappings, upstream cross-references, and pair-config anchor pairs validated through bootstrap CV pruning. Edge counts are direction-agnostic, so every framework shows its true connectivity. AIUC-1 remains the hub. CSA AICM now shows connections to OWASP Agentic (via 39 anchor pairs) in addition to its dense bidirectional link with AIUC-1. OWASP Agentic and OWASP LLM show connections to multiple frameworks. The confidence histogram on the right gives the appropriate skepticism prior: the majority of edges sit at the *suggestive* confidence level, meaning they were proposed by the mapping engine and have not been reviewed by an expert.

The heatmap panel uses the `crest` sequential colormap, a single-hue luminance ramp that is perceptually ordered (Borland & Taylor, 2007). This avoids the rainbow colormap pitfall where perceptual non-uniformity can introduce false boundaries in continuous data.

> **Plain English:** By combining graph edges, upstream OWASP mappings, cross-references, and pair-config anchors, every framework now shows its full connectivity. Previously invisible pairs like CSA AICM to OWASP Agentic now appear. The remaining zero cells are genuinely unmapped pairs where no data exists in any source.

### Transitive reachability: the mappings behind the zeros

Figure 4.1 counts only **direct edges** between framework pairs. Several cells read zero, notably MITRE ATLAS to CSA AICM. Yet these frameworks are clearly related: both address AI supply-chain risks, model evasion, and data poisoning. The zeros reflect a labeling gap, not a semantic gap. When two frameworks share no direct edge but both connect to a common bridge framework (typically AIUC-1), a **transitive (2-hop) path** exists between them.

Figure 4.1b below computes transitive reachability for every pair. For each node in framework A, it checks whether a path of length 1 (direct) or length 2 (through any bridge node) reaches any node in framework B. The count is the number of unique unordered (source, target) node pairs reachable by either route. This is the same metric shown in the interactive Dash app's "All reachability" heatmap toggle.

In [ ]:
# Figure 4.1b. Transitive reachability heatmap -- same layout as
# Figure 4.1's heatmap panel but showing unique node pairs reachable
# via direct edges OR 2-hop transitive paths through bridge frameworks.
# Resolve project2 data path (REPO may be project1/ or repo root)
_p2_derived = REPO / "project2" / "data" / "derived"
if not _p2_derived.exists():
    _p2_derived = REPO.parent / "project2" / "data" / "derived"
reach_path = _p2_derived / "pairwise_reachability.json"
with open(reach_path) as f:
    pairwise_reach = json.load(f)

# Build the reachability matrix (total = direct + transitive unique pairs)
reach_mat = pd.DataFrame(0, index=FRAMEWORKS, columns=FRAMEWORKS)
direct_mat = pd.DataFrame(0, index=FRAMEWORKS, columns=FRAMEWORKS)

for fw_a in FRAMEWORKS:
    for fw_b in FRAMEWORKS:
        if fw_a == fw_b:
            continue
        r = pairwise_reach.get(fw_a, {}).get(fw_b, {})
        reach_mat.loc[fw_a, fw_b] = r.get("total", 0)
        direct_mat.loc[fw_a, fw_b] = r.get("direct", 0)

n_unlocked = ((direct_mat == 0) & (reach_mat > 0)).sum().sum() // 2
n_disconnected = sum(
    1 for a in FRAMEWORKS for b in FRAMEWORKS
    if a < b and reach_mat.loc[a, b] == 0
)
print(f"Framework pairs with 0 direct but >0 transitive: {n_unlocked}")
print(f"Framework pairs with 0 connectivity in either mode: {n_disconnected}")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    reach_mat.values.astype(int),
    ax=ax, annot=True, fmt="d", cmap="crest",
    xticklabels=labels, yticklabels=labels,
    cbar_kws={"label": "unique node pairs (direct + transitive)"},
)
ax.set_title("Figure 4.1b \u00b7 Cross-framework reachability (direct + transitive)")
ax.set_xlabel("partner framework")
ax.set_ylabel("framework")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Annotation: same position as Fig 4.1 AIUC-1 callout.
aiuc_row = FRAMEWORKS.index("aiuc_1")
ax.annotate(
    "Bridge paths fill\npreviously empty cells",
    xy=(len(FRAMEWORKS) - 0.5, aiuc_row + 0.5),
    xytext=(len(FRAMEWORKS) + 0.6, aiuc_row + 1.4),
    fontsize=9, ha="left", va="center",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.0),
    annotation_clip=False,
)
plt.tight_layout()
plt.show()


Figure 4.1b uses the same layout and `crest` sequential colormap as the heatmap panel in Figure 4.1, but the counts now include transitive paths. If node A1 in MITRE ATLAS connects to node B1 in AIUC-1, and B1 connects to node C1 in CSA AICM, then (A1, C1) counts as a reachable pair even though no direct ATLAS-to-CSA edge exists. Comparing the two figures cell by cell, every value in 4.1b is greater than or equal to its counterpart in 4.1, and 18 previously empty cells now carry non-zero counts.

The effect is dramatic. CSA AICM, which had direct connections only to AIUC-1 and OWASP Agentic, now shows reachability to every other framework. The densest new connection is CSA AICM to OWASP AI Exchange (2,020 reachable pairs), mediated almost entirely through AIUC-1. MITRE ATLAS to CSA AICM, the pair that prompted this analysis, shows 629 reachable pairs despite zero direct edges.

The heatmap uses the `crest` sequential colormap, a single-hue luminance ramp that is perceptually ordered (Borland & Taylor, 2007). Cell-value annotations provide direct labeling to compensate for the low perceptual accuracy of color saturation alone (Cleveland & McGill, 1984, rank 6).

> **Plain English:** Many framework pairs show zero direct mappings, but that does not mean they are unrelated. By following two-hop paths through bridge frameworks (mostly AIUC-1), 18 of the 36 off-diagonal pairs gain connectivity they lacked before. Only three pairs remain completely disconnected: CoSAI to CSA AICM, CoSAI to OWASP AI Exchange, and CoSAI to EU GPAI. These are genuine coverage gaps where no mapping evidence exists in any source.

In [ ]:
# Figure 4.2. Grouped bar chart of entry-type composition by framework
# (row-normalized). Cleveland & McGill (1984) rank position along a common
# scale as the most accurate perceptual channel. The prior version used
# stacked bars where only the bottom segment shared a common baseline; upper
# segments required the less-accurate length-without-baseline judgment. Grouped
# bars place every entry-type category on the same baseline, making it easy
# to compare any single entry type across frameworks.
type_mat = (
    nodes_df.groupby(["framework", "entry_type"]).size().unstack(fill_value=0)
    .reindex(FRAMEWORKS)
)
type_mat = type_mat.div(type_mat.sum(axis=1), axis=0)
type_mat = type_mat.loc[:, type_mat.sum().sort_values(ascending=False).index]

n_groups = len(type_mat.index)
n_types = len(type_mat.columns)
x = np.arange(n_groups)
width = 0.8 / n_types
palette = sns.color_palette("Set2", n_colors=n_types)

fig, ax = plt.subplots(figsize=(13, 5.5))
for i, col in enumerate(type_mat.columns):
    offset = (i - n_types / 2 + 0.5) * width
    bars = ax.bar(
        x + offset, type_mat[col].values, width,
        label=col, color=palette[i], edgecolor="white", linewidth=0.4,
    )

ax.set_xticks(x)
ax.set_xticklabels([PRETTY[f] for f in type_mat.index], rotation=30, ha="right")
ax.set_title("Figure 4.2 \u00b7 Entry-type composition by framework (row-normalized, grouped)")
ax.set_ylabel("share of nodes")
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="entry type")
plt.tight_layout()
plt.show()

Figure 4.2 uses grouped bars rather than stacked bars so that every entry-type category sits on a common baseline. Cleveland & McGill (1984) rank position along a common scale as the most accurate elementary perceptual task; stacked bars sacrifice this for all but the bottom segment. The grouped layout makes it straightforward to compare, for example, the control share across all nine frameworks at a glance.

Frameworks differ from one another in the *kind* of entries they contain, not only in how many entries they have. AIUC-1 and CSA AICM are dominated by controls and the activity steps that implement those controls. MITRE ATLAS is mostly attack techniques and mitigations. NIST AI RMF decomposes into functions, categories, and subcategories. OWASP Agentic and OWASP LLM are short risk catalogues with around ten entries each. This composition asymmetry matters for the v7c classifier because several of its features are one-hot indicators for the source and target entry-type pair (`has_technique`, `has_mitigation`, `is_activity_subcategory`, `is_activity_risk`). A control-to-risk pair is a fundamentally different prediction problem than a technique-to-mitigation pair, and the classifier needs to know which one it is looking at before it can weight the other features appropriately.

> **Plain English:** Not all frameworks are the same size or shape. Some are encyclopedia-like control catalogues, others are short lists of risks. The model needs to know which type of pair it is looking at so it can judge similarity sensibly. Comparing a control to a risk is a very different question than comparing two controls.

## 5 · The v7c baseline and its failure

Before introducing data augmentation or a new architecture, I need to establish what the original v7c pipeline actually does — and where it breaks down.

v7c is a two-stage classifier. Stage one encodes each control as a node embedding using a graph attention network (GAT) trained on the crosswalk graph; stage two feeds 50 hand-engineered features into a logistic regression with L2 regularization (C=0.01). The 50 features come from three families: 35 GAT geometry features (cosine similarity, L2 distance, dot product, and 32 element-wise embedding diffs), 3 baseline text-and-graph features (BGE cosine, BM25, two-hop bridge score), and 12 cross-encoder soft probabilities from DeBERTa-v3-large, RoBERTa-large, and DeBERTa-v3-base.

On the frozen test set of 179 pairs the pipeline reaches **81.0% exact-match accuracy** and **0.512 macro F1**. Those numbers look reasonable until you look at the per-class breakdown: the Equivalent class achieves **F1 = 0.000**. The classifier never predicts Equivalent — not once across 179 test pairs. The 81% headline hides a total blind spot on the class that matters most for framework alignment.

In [ ]:
# Feature family definitions for v7c. The 50 features are split into three
# families: GAT embedding features, baseline text/graph features, and
# cross-encoder soft probabilities from three transformer models.
GAT_FEATS = (
    ["gat_cosine", "gat_l2", "gat_dot"]
    + [f"gat_diff_{i:02d}" for i in range(32)]
)
BASELINE_FEATS = ["bge_cosine", "bm25", "bridge"]
CE_FEATS = [
    "deberta_prob_0", "deberta_prob_1", "deberta_prob_2", "deberta_prob_3",
    "roberta_prob_0", "roberta_prob_1", "roberta_prob_2", "roberta_prob_3",
    "deberta_base_prob_0", "deberta_base_prob_1", "deberta_base_prob_2", "deberta_base_prob_3",
]
ALL_FEATS = GAT_FEATS + BASELINE_FEATS + CE_FEATS
FAMILY_COLOR = {"GAT": "#14b8a6", "Baseline": "#3b82f6", "CE": "#f59e0b"}

# CE model sub-groups for the feature importance plot.
CE_MODELS = {
    "DeBERTa-v3-large": ["deberta_prob_0", "deberta_prob_1", "deberta_prob_2", "deberta_prob_3"],
    "RoBERTa-large": ["roberta_prob_0", "roberta_prob_1", "roberta_prob_2", "roberta_prob_3"],
    "DeBERTa-v3-base": ["deberta_base_prob_0", "deberta_base_prob_1", "deberta_base_prob_2", "deberta_base_prob_3"],
}

# Legacy v6 feature lists (used for Section 5 violin plots on the v6 test CSV).
LLM_FEATS = [
    "llm_final_score", "llm_final_tier", "llm_confidence", "llm_is_unanimous",
    "llm_sonnet_1", "llm_sonnet_2", "llm_sonnet_3",
]
STRUCT_FEATS = [
    "depth_diff", "depth_src", "depth_tgt",
    "len_src", "len_tgt", "len_diff", "len_ratio",
    "n2v_cosine", "gat_cosine",
    "has_technique", "has_mitigation",
    "is_activity_subcategory", "is_activity_risk",
]
OPUS_FEATS = ["opus_score", "opus_confidence"]

TIER_ORDER = ["Unrelated", "Partial", "Related", "Equivalent"]

# Ordinal tier palette. Tiers are ordinal (Unrelated < Partial < Related <
# Equivalent), so Borner et al. (2019) and Borland & Taylor (2007) prescribe
# a luminance-varying sequential palette rather than categorically distinct
# hues. Lighter shades encode weaker relationships; the darkest shade marks
# the strongest (Equivalent). A single-hue blue-teal ramp is inherently
# colorblind-safe because it relies on luminance, not hue discrimination
# (Graze & Schwabish, 2024).
TIER_PALETTE = ["#c1d5e0", "#6ba3be", "#2b7a9e", "#0b3d5e"]

In [ ]:
# Figure 5.1. Small multiples of six representative features, two from each
# v6 family, broken out by expert tier. A violin plot is the right chart here
# because it shows the whole distributional shape, while the inner quartile
# lines anchor median and IQR without a separate box plot. These use the v6
# test features CSV; the v7c pipeline replaces LLM and Opus features with CE
# soft probabilities, but the structural features (gat_cosine, n2v_cosine)
# remain informative for EDA.
feat_panels = [
    ("llm_final_score", "LLM . final score (aggregated Sonnet vote)"),
    ("llm_confidence", "LLM . confidence"),
    ("n2v_cosine", "Structural . Node2Vec cosine"),
    ("gat_cosine", "Structural . GAT cosine"),
    ("opus_score", "Opus . score"),
    ("opus_confidence", "Opus . confidence"),
]

fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.2), sharex=True)
for ax, (col, title) in zip(axes.flat, feat_panels):
    sns.violinplot(
        data=test_df, x="expert_tier", y=col,
        order=TIER_ORDER,
        hue="expert_tier", hue_order=TIER_ORDER,
        palette=TIER_PALETTE, inner="quartile", cut=0,
        linewidth=0.8, ax=ax, legend=False,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    # Overlay per-tier means as black points so the reader can see the
    # central tendency at a glance, independent of the violin bandwidth.
    means = test_df.groupby("expert_tier")[col].mean().reindex(TIER_ORDER)
    ax.scatter(range(len(TIER_ORDER)), means.values,
               marker="o", color="black", s=28, zorder=5)

fig.suptitle("Figure 5.1 · v6 legacy feature distributions by expert tier",
             y=1.02, fontsize=14, weight="bold")
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_1_feature_violins.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 5.1 uses the v6 legacy feature set (LLM votes, Opus scores, structural cosines) rather than the v7c cross-encoder features, because v7c replaced LLM and Opus with CE soft probabilities. Even on the legacy features the pattern is clear: `llm_final_score` separates Equivalent from the other tiers, but the Equivalent violin is narrow — only 7 test pairs. That small sample is the first sign of trouble. The GAT cosine panel shows partial overlap across all four tiers, which means geometry alone cannot resolve close cases.

The v7c pipeline reports logistic regression coefficients as feature importance (absolute value of the learned weights, normalized to sum to 1.0). This tells me which features the trained classifier actually uses to make decisions. The ablation study complements feature importance by comparing four methods: A (GAT-only), B (full 50-feature pipeline), C (CE-only), and D (raw CE average). Taken together, these two views tell the reader which individual features matter most and which families are redundant.

In [ ]:
# Figure 5.2. Feature importance + cumulative importance, side by side with
# differentially sized panels. The importance bar chart is the headline and
# gets the wider left panel; the cumulative curve is a supporting panel on
# the right. Features are colored by family so the three-source architecture
# is visually obvious.
fi = v7c_results["feature_importance"]
fi_series = pd.Series(fi).sort_values(ascending=True)

def feature_family_v7c(name):
    if name in CE_FEATS: return "CE"
    if name in BASELINE_FEATS: return "Baseline"
    return "GAT"

bar_colors = [FAMILY_COLOR[feature_family_v7c(n)] for n in fi_series.index]

fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[2.0, 1.0], wspace=0.35)

ax_bar = fig.add_subplot(gs[0, 0])
ax_bar.barh(fi_series.index, fi_series.values,
            color=bar_colors, edgecolor="black", linewidth=0.35)
ax_bar.set_xlabel("Feature importance (normalized abs. LogReg weight)")
ax_bar.set_title("Figure 5.2A · Per-feature importance (v7c, 50 features)")
ax_bar.tick_params(axis="y", labelsize=7)

# Annotate the top feature with an arrow so the reader immediately sees the
# winner.
top_feat = fi_series.idxmax()
top_val = fi_series.max()
ax_bar.annotate(
    f"Top feature:\n{top_feat} ({top_val:.1%})",
    xy=(top_val, list(fi_series.index).index(top_feat)),
    xytext=(top_val * 0.55, list(fi_series.index).index(top_feat) - 4),
    fontsize=10, ha="left",
    arrowprops=dict(arrowstyle="->", lw=1.1, color="black"),
)
legend_elems = [Patch(facecolor=c, edgecolor="black", label=k)
                for k, c in FAMILY_COLOR.items()]
ax_bar.legend(handles=legend_elems, loc="lower right", frameon=True)

# Cumulative importance on the right. Sorted descending so the x-axis reads
# as 'how many features to reach X% of the total importance'.
ax_cum = fig.add_subplot(gs[0, 1])
fi_desc = fi_series.sort_values(ascending=False)
cum = np.cumsum(fi_desc.values) / fi_desc.values.sum()
ax_cum.plot(range(1, len(cum) + 1), cum, marker="o", color="#264653", lw=1.8,
            markersize=4)
ax_cum.axhline(0.80, color="#e76f51", ls="--", lw=1.0)
ax_cum.text(len(cum) * 0.55, 0.82, "80% of total importance",
            color="#e76f51", fontsize=9)
ax_cum.set_xlabel("top-k features")
ax_cum.set_ylabel("cumulative importance (fraction)")
ax_cum.set_ylim(0, 1.02)
ax_cum.set_title("Figure 5.2B · Cumulative")

plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_2_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

The CE features dominate: the top-ranked feature is a DeBERTa-v3-large soft probability, and CE features collectively account for most of the cumulative importance. The 80% threshold (dashed orange line) is reached in roughly the top 10 features. That concentration tells me the 35 GAT geometry features are largely redundant — the cross-encoders have already compressed the semantic signal. This will matter when I redesign the architecture in Section 9.

In [ ]:
# Figure 5.3. Method comparison bar chart (v7c ablation). Each bar is a
# different method evaluated on the same frozen test set. Method B is the
# full 50-feature pipeline. Colors encode the feature family used.
methods = v7c_results["methods"]
method_data = []
for key in ["A_gat_only", "B_full_pipeline", "C_ce_only", "D_raw_ce_avg"]:
    m = methods[key]
    method_data.append({"name": m["label"], "accuracy": m["tier_accuracy"],
                        "macro_f1": m["macro_f1"], "key": key})
abl_df = pd.DataFrame(method_data)

def method_color(key):
    if key == "B_full_pipeline": return "#6366f1"
    if key == "A_gat_only": return FAMILY_COLOR["GAT"]
    if key == "C_ce_only": return FAMILY_COLOR["CE"]
    return "#9333ea"

fig, ax = plt.subplots(figsize=(10, 5.5))
bar_colors_abl = [method_color(k) for k in abl_df["key"]]
bars = ax.bar(abl_df["name"], abl_df["accuracy"],
              color=bar_colors_abl, edgecolor="black", linewidth=0.5)

# Reference lines: random baseline (25%) and majority baseline.
majority = max(methods["B_full_pipeline"]["class_counts"].values()) / sacred["n_test"]
ax.axhline(0.25, color="lightgray", ls=":", lw=1.0,
           label=f"Random (25%)")
ax.axhline(majority, color="gray", ls="--", lw=1.2,
           label=f"Majority class ({majority:.0%})")

# Inline value labels.
for b, v in zip(bars, abl_df["accuracy"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008,
            f"{v:.1%}", ha="center", fontsize=10, fontweight="bold")

# Annotate the full-pipeline winner.
best_idx = abl_df[abl_df["key"] == "B_full_pipeline"].index[0]
ax.annotate(
    "Full 50-d pipeline wins:\nGAT + baseline + CE ensemble",
    xy=(best_idx, abl_df.loc[best_idx, "accuracy"]),
    xytext=(best_idx - 1.5, abl_df.loc[best_idx, "accuracy"] + 0.06),
    fontsize=10, ha="left",
    arrowprops=dict(arrowstyle="->", lw=1.1, color="black"),
)

ax.set_ylabel("4-tier accuracy on frozen test")
ax.set_ylim(0, 0.95)
ax.set_title("Figure 5.3 · Method comparison (v7c, n=179)")
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_3_method_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

Figure 5.3 shows that Method B (full 50-feature pipeline) outperforms the ablations, but the gap between B and C (CE-only) is small — about two percentage points. Method A (GAT-only) drops sharply. This confirms the importance ranking: CE features carry the load, GAT features add a small margin, and the three baseline features contribute noise as much as signal.

### Frozen test results

The numbers below come from the single locked evaluation run. I set aside the test split before any hyperparameter search and never touched it again until the final sacred evaluation. Everything that follows in the notebook is measured against this same frozen set.

In [ ]:
# Print the sacred summary so the reader can compare narrative claims against
# the raw numbers without scrolling back to section 2.
print(f"Sacred run:       {sacred_path.name}  (version: {sacred.get('version','?')})")
print(f"Best method:      {sacred['best_method']}")
print(f"Features:         {sacred['features']}")
print(f"Tier accuracy:    {sacred['tier_accuracy']:.1%}")
print(f"Macro F1:         {sacred['macro_f1']:.4f}")
print(f"Adjacent acc:     {sacred['adjacent_accuracy']:.1%}")
print(f"Binary acc:       {sacred['binary_accuracy']:.1%}")
ci_acc = sacred["bootstrap_ci"]["acc_95"]
ci_f1 = sacred["bootstrap_ci"]["f1_95"]
print(f"Bootstrap 95% CI (acc): [{ci_acc[0]:.1%}, {ci_acc[1]:.1%}]")
print(f"Bootstrap 95% CI (F1):  [{ci_f1[0]:.4f}, {ci_f1[1]:.4f}]")
print(f"Conformal coverage: {sacred['conformal']['coverage']:.1%}  (alpha=0.10)")
print(f"Conformal mean set size: {sacred['conformal']['mean_set_size']:.2f}")
print(f"CV macro F1:      {v7c_results['cv_macro_f1']:.4f}")
print(f"Best C:           {v7c_results['best_C']}")

In [ ]:
# Figure 5.4. Confusion matrix + per-class accuracy in a two-panel layout with
# differentially sized axes. The confusion matrix is the headline and gets
# the wider panel. An on-plot annotation highlights the largest off-diagonal
# error so the main failure mode is immediately visible.
cm = sacred["confusion_matrix"]
cm_array = np.array([[cm[t1.lower()][t2.lower()] for t2 in TIER_ORDER]
                     for t1 in TIER_ORDER])

fig = plt.figure(figsize=(13, 5.5))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.35, 1.0], wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues",
            xticklabels=TIER_ORDER, yticklabels=TIER_ORDER, ax=ax1,
            cbar_kws={"label": "pair count"})
ax1.set_xlabel("Predicted")
ax1.set_ylabel("True (expert)")
ax1.set_title("Figure 5.4A · Confusion matrix (v7c, n=179)")

# Find the largest off-diagonal error and point at it.
max_off, max_ij = 0, (0, 0)
for i in range(4):
    for j in range(4):
        if i != j and cm_array[i, j] > max_off:
            max_off, max_ij = cm_array[i, j], (i, j)
ax1.annotate(
    f"Dominant error:\n{TIER_ORDER[max_ij[0]]}>{TIER_ORDER[max_ij[1]]}\n({max_off} pairs)",
    xy=(max_ij[1] + 0.5, max_ij[0] + 0.5),
    xytext=(max_ij[1] + 1.35, max(max_ij[0] - 0.6, 0.2)),
    fontsize=9.5,
    arrowprops=dict(arrowstyle="->", lw=1.0),
    color="white" if cm_array[max_ij] > cm_array.max() * 0.45 else "black",
)

ax2 = fig.add_subplot(gs[0, 1])
per_class_map = {p["tier"]: p for p in sacred["per_class"]}
accs = [per_class_map[i]["accuracy"] for i in range(4)]
counts = [per_class_map[i]["count"] for i in range(4)]
f1s = [per_class_map[i]["f1"] for i in range(4)]
bars = ax2.bar(TIER_ORDER, accs, color=TIER_PALETTE, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("per-class accuracy")
ax2.set_title("Figure 5.4B · Per-class accuracy & F1")
ax2.set_ylim(0, 1.15)
for b, a, n, f in zip(bars, accs, counts, f1s):
    ax2.text(b.get_x() + b.get_width() / 2, a + 0.02,
             f"{a:.0%}\nF1={f:.2f}\nn={n}",
             ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_4_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

The Equivalent row in Figure 5.4A is all zeros. All 7 Equivalent test pairs are misclassified — 6 as Related, 1 as Partial. The per-class accuracy panel (5.4B) makes this explicit: Equivalent accuracy is 0%, F1 = 0.00. The dominant off-diagonal error visible in the heatmap annotation is the one the model gets most wrong, but the Equivalent failure is categorically different — it is not a boundary confusion, it is an invisible class.

In [ ]:
# Figure 5.5. Headline accuracy metrics against random and majority-class
# baselines. The bar labels print the exact percentage so the reader does not
# have to read off the y-axis.
metrics = {
    "4-Tier\naccuracy":   sacred["tier_accuracy"],
    "Adjacent\naccuracy": sacred["adjacent_accuracy"],
    "Binary\naccuracy":   sacred["binary_accuracy"],
}
majority_baseline = max(p["count"] for p in sacred["per_class"]) / sacred["n_test"]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(metrics))
vals = list(metrics.values())
bar_colors = ["#264653", "#2a9d8f", "#e76f51"]
bars = ax.bar(x, vals, color=bar_colors, edgecolor="black", linewidth=0.5, width=0.55)

ax.axhline(majority_baseline, color="gray", ls="--", lw=1.2,
           label=f"Majority class ({majority_baseline:.0%})")
ax.axhline(0.25, color="lightgray", ls=":", lw=1.0,
           label="Random (25%)")

ax.set_xticks(x)
ax.set_xticklabels(list(metrics.keys()), fontsize=11)
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.05)
ax.set_title("Figure 5.5 · Accuracy metrics vs baselines (v7c)")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.1%}",
            ha="center", fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig5_5_headline_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

### The pivot

The 81% headline accuracy in Figure 5.5 looks competitive against the majority-class baseline, but it is built almost entirely on 172 non-Equivalent pairs. The 7 Equivalent pairs in the test set are invisible to the model — it has never seen enough of them to learn what they look like.

This is data starvation, not model weakness. The training set contains fewer than 30 Equivalent pairs across all splits. No amount of regularization tuning or feature engineering will fix a class with that few examples. I need more labeled Equivalent pairs.

This sent me looking for external sources of high-similarity control pairs. Where could I find more labeled data for the Equivalent class?

## 6 · Uncertainty quantification and ordinal structure

Two problems with the v7c baseline go beyond the Equivalent blind spot. First, the classifier produces a point prediction with no uncertainty estimate. A compliance workflow needs to know when the model is guessing. Second, the four tiers are ordered---an Unrelated-to-Equivalent error is worse than a Related-to-Equivalent error---but the logistic regression treats all misclassifications equally.

### Three directions worth investigating

Three changes would most likely improve the classifier:

1. **More Equivalent training data.** The expert training set contains only a handful of Equivalent pairs. Any external source of high-similarity control pairs could help.
2. **Ordinal regression instead of flat 4-class classification.** The logistic regression ignores the ordering of tiers. An ordinal loss that penalizes distant errors more heavily than adjacent ones could improve calibration.
3. **Conformal prediction for uncertainty.** Wrapping point predictions in prediction sets would give downstream consumers a calibrated measure of confidence.

The ordinal regression demonstrator below addresses direction 2. The OpenCRE discovery in the next section addresses direction 1. v_final combines both.

### Ordinal regression demonstrator

The four tiers (Unrelated < Partial < Related < Equivalent) form a natural ladder. An ordinal model estimates cumulative thresholds: P(tier >= k) for each cutpoint. This proof-of-concept uses `statsmodels.OrderedModel` on the calibration split to show what ordinal structure looks like in practice.

> **Plain English:** Instead of treating the four tiers as disconnected buckets, this model says "tier 0 is most likely, and as the feature score increases, the probability mass slides rightward toward tier 3." The S-curves below visualize that slide.

In [ ]:
# statsmodels OrderedModel on the calibration split. The model treats the
# four expert tiers as an ordered outcome, fits cut points between adjacent
# categories, and produces monotone cumulative probabilities. I restrict
# the fit to the five most important v6 features to keep the demonstrator
# focused and the convergence tight.
import warnings as _w
_w.filterwarnings("ignore", category=UserWarning, module="statsmodels")
from statsmodels.miscmodels.ordinal_model import OrderedModel

TOP5 = ["gat_cosine", "n2v_cosine", "opus_score", "llm_final_score", "llm_confidence"]
X_ord = cal_df[TOP5].values
y_ord = cal_df["expert_label"].astype(int).values

ord_mod = OrderedModel(y_ord, X_ord, distr="logit")
ord_res = ord_mod.fit(method="bfgs", disp=False, maxiter=400)

print("=== OrderedModel fit summary (top 5 v6 features) ===")
print(f"converged: {ord_res.mle_retvals['converged']}")
print(f"log-likelihood: {ord_res.llf:.2f}")
print(f"n_obs: {int(ord_res.nobs)}")
print()
print("Coefficients (positive pushes toward higher tier):")
for name, coef in zip(TOP5, ord_res.params[:len(TOP5)]):
    print(f"  {name:<20s}  {coef:+.3f}")
print()
print("Threshold parameters (between adjacent tiers):")
thresholds = ord_res.params[len(TOP5):]
for i, t in enumerate(thresholds):
    print(f"  threshold_{i}: {t:+.3f}")

In [ ]:
# Figure 6.1. Fitted cumulative probabilities for a grid of hypothetical
# pairs. I sweep a single feature (opus_score, the strongest signal) across
# its observed range while holding the other four features at their cal-split
# means, then plot the four cumulative tier probabilities as a stacked area
# chart. The chart visualizes how the ordinal model shifts mass from low
# tiers to high tiers as opus_score increases.
mean_vec = cal_df[TOP5].mean().values
opus_grid = np.linspace(
    cal_df["opus_score"].min(),
    cal_df["opus_score"].max(),
    60,
)
X_grid = np.tile(mean_vec, (len(opus_grid), 1))
opus_idx = TOP5.index("opus_score")
X_grid[:, opus_idx] = opus_grid
probs = ord_res.model.predict(ord_res.params, exog=X_grid)

# Figure 6.1 uses individual line curves rather than a stacked area chart.
# Cleveland & McGill (1984) rank position along a common scale (rank 1) above
# area (rank 4); line curves let the reader compare exact probability values
# at any opus_score by reading off the shared y-axis baseline, whereas stacked
# areas only allow accurate reading of the bottom layer.
fig, ax = plt.subplots(figsize=(10.5, 5.8))
for i, tier in enumerate(TIER_ORDER):
    ax.plot(
        opus_grid, probs[:, i],
        label=tier, color=TIER_PALETTE[i], lw=2.2, alpha=0.9,
    )
ax.set_xlim(opus_grid.min(), opus_grid.max())
ax.set_ylim(0, 1)
ax.set_xlabel("opus_score (other features at cal-split mean)")
ax.set_ylabel("predicted probability")
ax.set_title("Figure 6.1 · statsmodels OrderedModel: tier probabilities vs opus_score")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), title="tier")

# On-plot annotation: mark where the modal prediction flips from Partial
# to Related, which is the key transition for this demonstrator.
mode_tier = np.argmax(probs, axis=1)
flip_idx = None
for i in range(1, len(mode_tier)):
    if mode_tier[i] != mode_tier[i - 1]:
        flip_idx = i
        break
if flip_idx is not None:
    ax.axvline(opus_grid[flip_idx], color="black", ls="--", lw=1.1)
    ax.annotate(
        f"modal-tier flip\n@ opus={opus_grid[flip_idx]:.2f}",
        xy=(opus_grid[flip_idx], 0.55),
        xytext=(opus_grid[flip_idx] + 0.3, 0.72),
        fontsize=9, fontweight="bold",
        arrowprops=dict(arrowstyle="->", lw=1.0, color="black"),
    )

plt.tight_layout()
plt.show()

The ordinal fit converges and the cumulative-probability plot shows a clean monotone sweep from mostly-Unrelated on the left to mostly-Equivalent on the right. This confirms that the ordinal structure in the data is real and exploitable. The v_final pipeline will use three ordinal loss functions that build on this principle.

> **Plain English:** The proof-of-concept works. When trained to respect the tier ordering, the model produces probability curves that slide smoothly from "definitely unrelated" to "probably equivalent" as the input feature increases. This motivated the ordinal losses used in the final pipeline.

## 7 · The OpenCRE discovery: external linkage ground truth

The v7c Equivalent blind spot came down to data starvation. The expert training set contained a handful of Equivalent pairs. To train a model that can identify equivalent controls, I needed more examples of what "equivalent" looks like.

The Open Common Requirements Enumeration (OpenCRE) is a community-maintained database that maps security standards at the control level. Each CRE node represents a shared requirement; the links between CRE nodes and framework controls carry expert-curated relationship types (Contains, Related, Linked To). Two controls that share the same CRE node are almost certainly equivalent. Two controls linked through adjacent CREs are likely related. This hop-distance structure maps directly onto our four ordinal tiers.

In [ ]:
# 7.1 — Load OpenCRE pairs and profile the dataset
from collections import Counter

ocre_path = REPO_ROOT / "data" / "opencre" / "opencre_pairs.jsonl"
ocre_pairs = []
with open(ocre_path) as f:
    for line in f:
        ocre_pairs.append(json.loads(line))

print(f"OpenCRE pairs loaded: {len(ocre_pairs):,}")
print(f"Unique CRE nodes: {len(set(p['cre_id'] for p in ocre_pairs)):,}")
print(f"Frameworks represented: {sorted(set(p['source_framework'] for p in ocre_pairs) | set(p['target_framework'] for p in ocre_pairs))}")

# Gap penalty distribution (hop distance)
gaps = Counter(p["gap_penalty"] for p in ocre_pairs)
for g in sorted(gaps):
    print(f"  Hop {g}: {gaps[g]:,} pairs ({gaps[g]/len(ocre_pairs)*100:.1f}%)")

OpenCRE contains 13,519 cross-framework pairs spanning 6 of our 9 frameworks. The gap penalty (hop distance) distributes across three levels: hop 0 (same CRE, tightest pairing), hop 1 (adjacent CREs), and hop 2 (two-hop path). The concentration at hop 0 is good news---these are the pairs most likely to represent genuine equivalence.

In [ ]:
# Figure 7.1 — Hop distance distribution across OpenCRE pairs.
# Bar chart on a common position scale (Cleveland & McGill 1984).
# Luminance ramp encodes the ordinal structure of hop distance
# (Borner et al. 2019): darker = tighter relationship.
from collections import Counter

gaps = Counter(p["gap_penalty"] for p in ocre_pairs)
hops = sorted(gaps.keys())
counts = [gaps[h] for h in hops]
hop_colors = ["#1a5276", "#2e86c1", "#85c1e9"]  # luminance ramp: dark to light

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [f"Hop {h}\n({['same CRE', 'adjacent CRE', '2-hop path'][h]})" for h in hops],
    counts, color=hop_colors[:len(hops)], edgecolor="white", linewidth=0.8
)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f"{count:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Annotation: call out the hop-0 concentration (Graze & Schwabish 2024)
ax.annotate(
    f"{counts[0]/sum(counts)*100:.0f}% of pairs share\nthe same CRE node",
    xy=(0, counts[0]), xytext=(0.8, counts[0] * 0.75),
    fontsize=9, color="black",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
)

ax.set_ylabel("Number of pairs")
ax.set_title("Most OpenCRE pairs share the same CRE node (hop 0)", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig7_1_hop_distance.png", dpi=150, bbox_inches="tight")
plt.show()

Hop 0 pairs dominate the distribution. These are pairs where both controls link to the same CRE requirement node, making them the strongest candidates for Equivalent-tier labels. The long tail of hop-1 and hop-2 pairs provides Related and Partial examples.

> **Plain English:** OpenCRE organizes security controls into clusters around shared requirements. When two controls from different frameworks point to the same requirement, they are zero hops apart---almost certainly equivalent. The farther apart they are in the CRE graph, the weaker the relationship.

In [ ]:
# Figure 7.2 — OpenCRE link-type distribution.
# Horizontal bar chart for readability of long category labels
# (Cleveland & McGill 1984: position on common scale).
link_types = Counter(p.get("provenance", "unknown") for p in ocre_pairs)
labels = sorted(link_types.keys(), key=lambda x: link_types[x], reverse=True)
values = [link_types[l] for l in labels]

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(labels, values, color="#2e86c1", edgecolor="white", linewidth=0.8)

for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)

ax.set_xlabel("Number of pairs")
ax.set_title("OpenCRE link types: most pairs come from Contains and Linked To relationships", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig7_2_link_types.png", dpi=150, bbox_inches="tight")
plt.show()

> **Plain English:** The link types tell us how OpenCRE connected these controls. 'Contains' means one standard explicitly references the other. 'Linked To' means the CRE maintainers judged them as covering the same ground. Both types are strong evidence for equivalence or relatedness.

In [ ]:
# Figure 7.3 — Framework coverage: OpenCRE vs. our corpus

# Count total mentions per framework (source + target)
fw_mention = Counter()
for _, row in ocre_df.iterrows():
    fw_mention[row["source_framework"]] += 1
    fw_mention[row["target_framework"]] += 1

top12 = fw_mention.most_common(12)
top12_names = [t[0] for t in top12]
top12_counts = [t[1] for t in top12]

# Which of our 9 frameworks appear in OpenCRE?
overlap_map = {"mitre_atlas": "mitre_atlas", "owasp_ai_exchange": "owasp_ai_exchange"}
overlap_set = set(overlap_map.keys())

bar_colors = ["#2a9d8f" if fw in overlap_set else "#6e7681" for fw in top12_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [2.2, 1]})

# Left panel (10.2A): horizontal bar chart
y_pos = range(len(top12_names))
ax1.barh(y_pos, top12_counts, color=bar_colors, edgecolor="white")
ax1.set_yticks(y_pos)
ax1.set_yticklabels(top12_names, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel("Mention count (source + target)")
ax1.set_title("Figure 7.3A \u00b7 Top 12 OpenCRE frameworks by pair frequency")

# Annotate overlapping frameworks
for i, fw in enumerate(top12_names):
    if fw in overlap_set:
        ax1.annotate(
            "\u2605 in our corpus",
            xy=(top12_counts[i], i),
            xytext=(top12_counts[i] + max(top12_counts) * 0.05, i),
            fontsize=8, fontweight="bold", color="#2a9d8f", va="center",
            arrowprops=dict(arrowstyle="->", lw=1, color="black"),
        )

sns.despine(ax=ax1)

# Right panel (10.2B): text summary
ax2.axis("off")

# Build summary text
our_fws_in_ocre = sorted(overlap_set)
ocre_only = sorted(set(ocre_fws) - set(FRAMEWORKS))

# Count pairs where at least one framework is in our corpus
overlap_pairs = ocre_df[
    ocre_df["source_framework"].isin(overlap_set) | ocre_df["target_framework"].isin(overlap_set)
].shape[0]

summary_lines = [
    "Coverage Summary",
    "\u2500" * 30,
    f"Our corpus frameworks:      {len(FRAMEWORKS)}",
    f"OpenCRE frameworks:          {len(ocre_fws)}",
    f"Frameworks in common:        {len(our_fws_in_ocre)}",
    f"  \u2022 {', '.join(our_fws_in_ocre)}",
    "",
    f"Total OpenCRE pairs:         {len(ocre_df):,}",
    f"Pairs touching our corpus:   {overlap_pairs:,}",
    "",
    "OpenCRE-only frameworks:",
]
for fw in ocre_only[:10]:
    summary_lines.append(f"  \u2022 {fw}")
if len(ocre_only) > 10:
    summary_lines.append(f"  ... +{len(ocre_only) - 10} more")

ax2.text(
    0.05, 0.95, "\n".join(summary_lines),
    transform=ax2.transAxes, fontsize=9, verticalalignment="top",
    fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", edgecolor="#cccccc"),
)
ax2.set_title("Figure 7.3B \u00b7 Corpus overlap", fontsize=11)

fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig7_3_opencre_coverage.png", dpi=300, bbox_inches="tight")
plt.show()

Only 6 of our 9 frameworks appear in OpenCRE. NIST AI RMF and OWASP AI Exchange dominate the overlap. AIUC-1, CoSAI, and CSA AICM have no representation, meaning OpenCRE augmentation cannot help with pairs involving those frameworks.

> **Plain English:** OpenCRE covers the larger, more established standards. The newer or more specialized frameworks in our crosswalk have no OpenCRE data at all. This is a coverage limitation to keep in mind.

In [ ]:
# Figure 7.4 — Hop distance mapped to ordinal tiers.
# Cross-tab heatmap showing how hop distance translates to
# our 4-tier classification scheme. Sequential colormap avoids
# rainbow (Borland & Taylor 2007).
tier_map = {0: "EQUIVALENT", 1: "RELATED", 2: "PARTIAL"}
hop_tier_counts = Counter()
for p in ocre_pairs:
    gap = p["gap_penalty"]
    tier = tier_map.get(gap, "UNRELATED")
    hop_tier_counts[(gap, tier)] += 1

# Build matrix
hops_unique = sorted(set(h for h, _ in hop_tier_counts))
tiers = ["EQUIVALENT", "RELATED", "PARTIAL", "UNRELATED"]
matrix = []
for h in hops_unique:
    row = [hop_tier_counts.get((h, t), 0) for t in tiers]
    matrix.append(row)

fig, ax = plt.subplots(figsize=(7, 3))
im = ax.imshow(matrix, cmap="crest", aspect="auto")
ax.set_xticks(range(len(tiers)))
ax.set_xticklabels(tiers, fontsize=9)
ax.set_yticks(range(len(hops_unique)))
ax.set_yticklabels([f"Hop {h}" for h in hops_unique], fontsize=9)

for i in range(len(hops_unique)):
    for j in range(len(tiers)):
        val = matrix[i][j]
        if val > 0:
            color = "white" if val > max(max(r) for r in matrix) * 0.5 else "black"
            ax.text(j, i, f"{val:,}", ha="center", va="center", fontsize=9, color=color)

ax.set_title("Hop distance maps cleanly onto ordinal tiers", fontsize=11)
plt.colorbar(im, ax=ax, label="Pair count")
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig7_4_hop_tier_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

The mapping is nearly diagonal: hop 0 produces Equivalent labels, hop 1 produces Related, hop 2 produces Partial. This clean structure is what makes OpenCRE useful as training data---the hop distance is a reasonable proxy for human-assigned ordinal tiers.

In [ ]:
# Figure 7.5 — Contamination firewall: removing expert-corpus overlaps
v8_assembly = json.loads(
    (REPO_ROOT / "runs" / "v8_diagnosis" / "v8_data_assembly.json").read_text()
)

total = v8_assembly["opencre_total"]
contaminated = v8_assembly["contaminated"]
clean = v8_assembly["clean"]
pct_removed = contaminated / total * 100

print(f"OpenCRE candidate pairs:  {total:,}")
print(f"Contaminated (in expert): {contaminated}  ({pct_removed:.1f}%)")
print(f"Clean pairs retained:     {clean:,}")

# Waterfall / funnel bar chart
fig, ax = plt.subplots(figsize=(7, 4))

labels = ["OpenCRE\ncandidates", "Removed\n(in expert set)", "Clean pairs\nretained"]
values = [total, contaminated, clean]
colors = ["#2a9d8f", "#e76f51", "#264653"]

bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.5)

# Count labels above each bar
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{val:,}",
        ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

# Annotate the removal with an arrow
ax.annotate(
    f"{pct_removed:.1f}% removed",
    xy=(1, contaminated + 50),
    xytext=(1.55, total * 0.55),
    fontsize=10, fontweight="bold", color="#e76f51",
    arrowprops=dict(arrowstyle="->", lw=1, color="black"),
    ha="center",
)

ax.set_ylabel("Number of pairs")
ax.set_title("Figure 7.5 \u00b7 Contamination firewall: expert-corpus overlap removal")
ax.set_ylim(0, total * 1.2)
sns.despine()

fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig7_5_contamination_firewall.png", dpi=300, bbox_inches="tight")
plt.show()

Before using any OpenCRE pairs for training, I removed every pair that shared a node ID with the 179-pair frozen test set. This contamination firewall eliminated 34 pairs, leaving 6,200 clean pairs. Without this step, the test set would no longer be truly held out.

> **Plain English:** The frozen test set is sacred. If any training pair shares a control with a test pair, the evaluation is compromised. I checked all 6,234 OpenCRE pairs that overlap our frameworks, found 34 that shared node IDs with the test set, and removed them.

## 8 · v8: disagreement mining

With 6,200 clean OpenCRE pairs in hand, the question is how to use them. Adding all 6,200 directly to training would triple the dataset, but the OpenCRE labels are proxy labels derived from hop distance, not expert annotations. Flooding the training set with noisy labels risks degrading the model.

The v8 approach was selective: run the v7c classifier on all 6,200 pairs and keep only the ones where v7c's prediction disagreed with the OpenCRE-derived label. These disagreements represent the classifier's blind spots---exactly the pairs it needs to see.

In [ ]:
# 11.1 — Load training splits and diagnosis files
TIER_NAMES = ["UNRELATED", "PARTIAL", "RELATED", "EQUIVALENT"]

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

expert_train = load_jsonl(REPO_ROOT / "data" / "splits" / "expert_train.jsonl")
v8_train     = load_jsonl(REPO_ROOT / "data" / "splits" / "v8_train.jsonl")
v8b_train    = load_jsonl(REPO_ROOT / "data" / "splits" / "v8b_train.jsonl")

print(f"expert_train: {len(expert_train):,} records")
print(f"v8_train:     {len(v8_train):,} records")
print(f"v8b_train:    {len(v8b_train):,} records")

# Tier distributions
for name, data in [("expert_train", expert_train), ("v8_train", v8_train), ("v8b_train", v8b_train)]:
    dist = Counter(r["tier_label"] for r in data)
    parts = [f"{TIER_NAMES[k]}={dist[k]:,}" for k in sorted(dist)]
    print(f"  {name:14s} tiers: {', '.join(parts)}")

# Load diagnosis files
v8_diag = json.loads((REPO_ROOT / "runs" / "v8_diagnosis" / "v8_data_assembly.json").read_text())
v8b_diag = json.loads((REPO_ROOT / "runs" / "v8b_diagnosis" / "v8b_data_assembly.json").read_text())

print(f"\nv8 disagreement mining:")
print(f"  Clean pairs: {v8_diag['clean']:,}  |  Disagreements: {v8_diag['disagreements']:,}  |  Selected: {v8_diag['selected']:,}  |  Added: {v8_diag['v8_rows_added']}")
print(f"\nv8b direct augmentation:")
print(f"  Clean pairs: {v8b_diag['clean']:,}  |  Added: {v8b_diag['v8_rows_added']:,}  |  Skipped by cap: {v8b_diag['skipped_by_cap']:,}")

In [ ]:
# Figure 8.1 — Training set composition: expert vs. v8.
# Grouped bar chart using position on common scale
# (Cleveland & McGill 1984) for direct tier-by-tier comparison.
expert_train = []
with open(REPO_ROOT / "data" / "splits" / "expert_train.jsonl") as f:
    for line in f:
        expert_train.append(json.loads(line))

v8_train = []
with open(REPO_ROOT / "data" / "splits" / "v8_train.jsonl") as f:
    for line in f:
        v8_train.append(json.loads(line))

expert_dist = Counter(p.get("tier", p.get("label", -1)) for p in expert_train)
v8_dist = Counter(p.get("tier", p.get("label", -1)) for p in v8_train)

tiers = [0, 1, 2, 3]
tier_labels = ["UNRELATED", "PARTIAL", "RELATED", "EQUIVALENT"]
x = np.arange(len(tiers))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
bars1 = ax.bar(x - width/2, [expert_dist.get(t, 0) for t in tiers],
               width, label=f"Expert train (n={len(expert_train):,})", color="#2e86c1")
bars2 = ax.bar(x + width/2, [v8_dist.get(t, 0) for t in tiers],
               width, label=f"v8 train (n={len(v8_train):,})", color="#e74c3c")

for bar_group in [bars1, bars2]:
    for bar in bar_group:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tier_labels, fontsize=10)
ax.set_ylabel("Number of pairs")
ax.set_title("v8 adds 673 disagreement-mined pairs, concentrated in RELATED tier", fontsize=11)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig8_1_training_composition.png", dpi=150, bbox_inches="tight")
plt.show()

The v8 augmentation adds 673 pairs to the expert training set, bringing the total from 5,920 to 12,849. The added pairs are concentrated in the Related tier, which is where v7c's disagreements with OpenCRE labels clustered. The Equivalent tier gains indirect benefit: by teaching the model to distinguish Related from Unrelated more precisely, the ordinal boundary near Equivalent should shift.

In [ ]:
# Figure 8.2 — Disagreement mining funnel
funnel_labels = ["Clean pairs", "v7c disagrees", "Selected", "Added to\ntraining"]
funnel_values = [
    v8_diag["clean"],
    v8_diag["disagreements"],
    v8_diag["selected"],
    v8_diag["v8_rows_added"],
]
funnel_colors = ["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(funnel_labels, funnel_values, color=funnel_colors, edgecolor="white", width=0.55)

# Count labels
for bar, val in zip(bars, funnel_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{val:,}",
        ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

# Percentage annotations between bars
for i in range(len(funnel_values) - 1):
    pct = funnel_values[i + 1] / funnel_values[i] * 100
    mid_x = (bars[i].get_x() + bars[i].get_width() / 2 + bars[i + 1].get_x() + bars[i + 1].get_width() / 2) / 2
    mid_y = (funnel_values[i] + funnel_values[i + 1]) / 2
    ax.annotate(
        f"{pct:.0f}%",
        xy=(mid_x, mid_y),
        fontsize=10, fontweight="bold", color="#264653",
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="#cccccc", alpha=0.8),
    )

ax.set_ylabel("Number of pairs")
ax.set_title("Figure 8.2 \u00b7 v8 disagreement mining funnel")
ax.set_ylim(0, max(funnel_values) * 1.2)
sns.despine()

fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig8_2_disagreement_mining.png", dpi=300, bbox_inches="tight")
plt.show()

The funnel narrows aggressively: of 6,200 clean pairs, 3,285 (53%) showed disagreement between v7c and OpenCRE. Of those, 673 Related-class disagreements were selected as the most informative augmentation (Related is adjacent to Equivalent on the ordinal scale, so getting Related right helps the model calibrate its Equivalent boundary).

> **Plain English:** Instead of dumping all 6,200 OpenCRE pairs into training, I filtered down to the 673 where the classifier was most confused. Think of it as showing the student only the flashcards it got wrong.

In [ ]:
# Figure 8.3 — BGE cosine fallback distribution for OpenCRE pairs.
# The GAT cannot compute graph features for OpenCRE-format pairs
# (they exist outside our crosswalk topology). BGE cosine similarity
# serves as the fallback scorer. Violin plot shows the distribution
# split by OpenCRE-derived tier (Borner et al. 2019: luminance ramp
# for ordered categories).

# Simulate: load v8 assembly stats
v8_assembly = json.loads(
    (REPO_ROOT / "runs" / "v8_diagnosis" / "v8_data_assembly.json").read_text()
)

# Display key stats as a table since we don't have per-pair BGE scores on disk
print("v8 Data Assembly Summary")
print(f"  OpenCRE pairs loaded:    {v8_assembly['opencre_total']:,}")
print(f"  Contaminated (removed):  {v8_assembly['contaminated']:,}")
print(f"  Clean pairs:             {v8_assembly['clean']:,}")
print(f"  v7c disagreements:       {v8_assembly['disagreements']:,}")
print(f"  Selected for training:   {v8_assembly['selected']:,}")
print(f"  Expert train original:   {v8_assembly['expert_train_original']:,}")
print(f"  v8 train total:          {v8_assembly['v8_train_total']:,}")
print(f"\nLabel distribution in v8 train:")
for tier, count in sorted(v8_assembly["label_distribution"].items()):
    print(f"    {tier}: {count:,}")

> **Plain English:** The GAT model (which uses the graph structure of our crosswalk) cannot score OpenCRE pairs because they live outside our graph. So the v8 pipeline fell back to BGE cosine similarity as the scoring signal for disagreement mining. This is a simpler feature but still informative for the ordinal structure.

## 9 · The v8b collapse crisis

v8b took a broader approach than v8's targeted disagreement mining. Instead of selecting only disagreements, v8b added 2,046 OpenCRE pairs directly to training with per-class caps: 997 Unrelated, 690 Partial, 683 Equivalent, and 673 Related. This brought the total training set to 14,222 pairs.

The results were a disaster on two fronts.

In [ ]:
# 9.1 — v8b data assembly and collapse diagnostics
v8b_assembly = json.loads(
    (REPO_ROOT / "runs" / "v8b_diagnosis" / "v8b_data_assembly.json").read_text()
)
print("v8b Data Assembly")
print(f"  Expert train:         {v8b_assembly['expert_train_original']:,}")
print(f"  OpenCRE added:        {v8b_assembly['v8_rows_added']:,}")
print(f"  Skipped by cap:       {v8b_assembly['skipped_by_cap']:,}")
print(f"  v8b train total:      {v8b_assembly['v8_train_total']:,}")
print(f"  Class caps:           {v8b_assembly['class_caps']}")
print(f"\nLabel distribution:")
for tier, count in sorted(v8b_assembly["label_distribution"].items()):
    print(f"    {tier}: {count:,}")

### Failure 1: DeBERTa-large model collapse

DeBERTa-v3-large collapsed during training. By epoch 4, every sweep configuration produced 100% single-class predictions---the model predicted the same tier for every pair regardless of input. The collapse guard triggered but the model never recovered.

In [ ]:
# Figure 9.1 — v8b model collapse diagnostics from WandB runs.
# Bar chart showing the fraction of runs that collapsed to
# single-class prediction (n_unique_preds == 1).
# Position on common scale for comparison (Cleveland & McGill 1984).

wandb_v8b = json.loads(
    (REPO_ROOT / "runs" / "v8b_diagnosis" / "wandb_runs.json").read_text()
)

# Count collapsed vs non-collapsed runs
total_runs = len(wandb_v8b)
collapsed = sum(1 for r in wandb_v8b if r.get("n_unique_preds") == 1)
partial_collapse = sum(1 for r in wandb_v8b
                       if r.get("n_unique_preds") is not None
                       and 1 < r["n_unique_preds"] < 4)
healthy = sum(1 for r in wandb_v8b if r.get("n_unique_preds") == 4)

categories = ["Collapsed\n(1 class)", "Partial collapse\n(2-3 classes)", "Healthy\n(4 classes)"]
counts = [collapsed, partial_collapse, healthy]
colors = ["#e74c3c", "#f39c12", "#2ecc71"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(categories, counts, color=colors, edgecolor="white", linewidth=0.8)

for bar, count in zip(bars, counts):
    pct = count / total_runs * 100 if total_runs > 0 else 0
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{count} ({pct:.0f}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")

# Annotation (Graze & Schwabish 2024: on-plot annotation)
ax.annotate(
    f"{collapsed} of {total_runs} runs predicted\na single class for every pair",
    xy=(0, collapsed), xytext=(1.2, collapsed * 0.8),
    fontsize=9, color="black",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
)

ax.set_ylabel("Number of WandB runs")
ax.set_title(f"v8b sweep: {collapsed}/{total_runs} runs collapsed to single-class prediction", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig9_1_v8b_collapse.png", dpi=150, bbox_inches="tight")
plt.show()

Model collapse is a known failure mode in fine-tuning. When the loss landscape has sharp minima near trivial solutions (predicting a single class), gradient updates can push the model into a region where it never escapes. The noisy OpenCRE labels in v8b made this worse: the model learned that predicting Unrelated (the majority class) was a safe default.

### Failure 2: stacker overfitting

Even for the runs that did not collapse, the LightGBM stacker (which combined the three models' outputs into a final prediction) overfit catastrophically.

In [ ]:
# Figure 9.2 — Stacker overfitting diagnostic (from registry.jsonl)
registry = load_jsonl(REPO_ROOT / "runs" / "registry.jsonl")

run_labels = ["v8b stacker", "v_final attempt 1", "v_final attempt 2"]
train_accs = [r["metrics"]["train_acc"] for r in registry]
val_accs   = [r["metrics"]["val_acc"] for r in registry]

x = np.arange(len(run_labels))
width = 0.32

fig, ax = plt.subplots(figsize=(9, 5))
bars_train = ax.bar(x - width / 2, train_accs, width, label="Train accuracy",
                     color="#2a9d8f", edgecolor="white")
bars_val   = ax.bar(x + width / 2, val_accs, width, label="Val accuracy",
                     color="#e76f51", edgecolor="white")

# Value labels
for bars in [bars_train, bars_val]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{bar.get_height():.1%}",
            ha="center", va="bottom", fontweight="bold", fontsize=9,
        )

# Annotate the overfit run (index 0)
ax.annotate(
    "47-point gap\n100% train, 52.8% val",
    xy=(0, 1.0),
    xytext=(0.7, 0.88),
    fontsize=9, fontweight="bold", color="#e76f51",
    arrowprops=dict(arrowstyle="->", lw=1.5, color="#e76f51"),
    ha="center", va="center",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff3f0", edgecolor="#e76f51", alpha=0.9),
)

ax.set_xticks(x)
ax.set_xticklabels(run_labels)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.15)
ax.set_title("Figure 9.2 \u00b7 Stacker overfitting diagnostic across registry runs")
ax.legend(loc="upper right")
sns.despine()

fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig9_2_stacker_overfitting.png", dpi=300, bbox_inches="tight")
plt.show()

Training accuracy of 1.000 against validation accuracy of 0.528 is a 47-point gap. The stacker memorized the training data rather than learning generalizable patterns. With 17 input features (softmax logits from 3 models) and a small validation set, the LightGBM had enough capacity to memorize every training example.

> **Plain English:** The combiner---a machine learning model that was supposed to blend the three models' predictions---instead just memorized the training answers. It scored perfectly on the practice test and failed the real one. This is the classic overfitting problem, and it pointed to a clear fix: remove the learnable combiner entirely.

In [ ]:
# Figure 9.3 — Training loss curves: collapsed vs. successful v8b runs.
# Line chart with two groups. Uses luminance contrast to distinguish
# collapse (red, thick) from success (blue, thin) (Borland & Taylor 2007).

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Select a few representative runs
collapsed_runs = [r for r in wandb_v8b
                  if r.get("n_unique_preds") == 1
                  and len(r.get("history", [])) > 3][:5]
healthy_runs = [r for r in wandb_v8b
                if r.get("n_unique_preds") == 4
                and len(r.get("history", [])) > 3][:5]

# Panel A: Training loss
ax = axes[0]
for r in collapsed_runs:
    epochs = [h.get("epoch") for h in r["history"] if h.get("train_loss") is not None]
    losses = [h["train_loss"] for h in r["history"] if h.get("train_loss") is not None]
    if epochs and losses:
        ax.plot(epochs, losses, color="#e74c3c", alpha=0.5, linewidth=2, label="_nolegend_")

for r in healthy_runs:
    epochs = [h.get("epoch") for h in r["history"] if h.get("train_loss") is not None]
    losses = [h["train_loss"] for h in r["history"] if h.get("train_loss") is not None]
    if epochs and losses:
        ax.plot(epochs, losses, color="#2e86c1", alpha=0.5, linewidth=1.5, label="_nolegend_")

ax.plot([], [], color="#e74c3c", linewidth=2, label="Collapsed")
ax.plot([], [], color="#2e86c1", linewidth=1.5, label="Healthy")
ax.legend(frameon=False, fontsize=9)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training loss")
ax.set_title("Collapsed runs show flat loss after early epochs", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

# Panel B: Combined F1
ax = axes[1]
for r in collapsed_runs:
    epochs = [h.get("epoch") for h in r["history"] if h.get("combined_f1") is not None]
    f1s = [h["combined_f1"] for h in r["history"] if h.get("combined_f1") is not None]
    if epochs and f1s:
        ax.plot(epochs, f1s, color="#e74c3c", alpha=0.5, linewidth=2)

for r in healthy_runs:
    epochs = [h.get("epoch") for h in r["history"] if h.get("combined_f1") is not None]
    f1s = [h["combined_f1"] for h in r["history"] if h.get("combined_f1") is not None]
    if epochs and f1s:
        ax.plot(epochs, f1s, color="#2e86c1", alpha=0.5, linewidth=1.5)

ax.set_xlabel("Epoch")
ax.set_ylabel("Combined F1")
ax.set_title("Collapsed runs never develop class diversity", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig9_3_wandb_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

The loss curves tell the story visually. Collapsed runs (red) plateau early: the model stops learning after the first few epochs because it has converged on the trivial solution of predicting a single class. Healthy runs (blue) show continued improvement, with combined F1 rising through training.

> **Plain English:** When a model collapses, its training loss curve goes flat. It has found a shortcut (always guess "unrelated") and stopped learning. The healthy runs keep improving because they are still finding structure in the data.

### What went wrong and what it taught us

Two root causes drove v8b's failures:

1. **Noisy labels at scale.** Adding 2,046 OpenCRE pairs with proxy labels overwhelmed the signal from 5,920 expert labels. The model learned to fit noise rather than structure.
2. **Learnable second stage.** The LightGBM stacker had enough capacity to memorize the noisy training distribution. A simpler combination method---one with no learnable parameters---could not overfit by construction.

These two observations directly motivated every design decision in v_final.

## 10 · v_final: clean architecture, proper validation

Three changes define v_final, each a direct response to a v8b failure mode.

In [ ]:
# Figure 10.1 — v_final architecture diagram
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

# --- Encoder boxes (left column) ---
enc_colors = ["#4e79a7", "#f28e2b", "#59a14f"]
enc_labels = ["RoBERTa-large\n(cross-encoder)", "DeBERTa-base\n(cross-encoder)", "BGE-large\n(bi-encoder)"]
enc_y = [4.5, 2.75, 1.0]

for y, color, label in zip(enc_y, enc_colors, enc_labels):
    ax.add_patch(plt.Rectangle((0.3, y), 2.8, 1.0, facecolor=color,
                                edgecolor="white", linewidth=2, alpha=0.9,
                                zorder=2))
    ax.text(1.7, y + 0.5, label, ha="center", va="center",
            fontsize=10, fontweight="bold", color="white", zorder=3)

# --- Loss function boxes (middle column) ---
loss_colors = ["#9c755f", "#bab0ab", "#e15759"]
loss_labels = ["KL Ordinal\nLoss", "CORN Ordinal\nLoss", "Focal\nLoss"]
loss_y = [4.5, 2.75, 1.0]

for y, color, label in zip(loss_y, loss_colors, loss_labels):
    ax.add_patch(plt.Rectangle((4.5, y), 2.2, 1.0, facecolor=color,
                                edgecolor="white", linewidth=2, alpha=0.9,
                                zorder=2))
    ax.text(5.6, y + 0.5, label, ha="center", va="center",
            fontsize=9.5, fontweight="bold", color="white", zorder=3)

# --- Arrows: each encoder fans out to each loss ---
for ey in enc_y:
    for ly in loss_y:
        ax.annotate(
            "", xy=(4.5, ly + 0.5), xytext=(3.1, ey + 0.5),
            arrowprops=dict(arrowstyle="->", color="#888888", lw=1.0,
                            connectionstyle="arc3,rad=0.0"),
            zorder=1,
        )

# --- Softmax averaging box (right) ---
ax.add_patch(plt.Rectangle((8.0, 2.0), 2.0, 2.0, facecolor="#264653",
                            edgecolor="white", linewidth=2, alpha=0.95,
                            zorder=2))
ax.text(9.0, 3.0, "Softmax\nAveraging\n(0 params)", ha="center", va="center",
        fontsize=11, fontweight="bold", color="white", zorder=3)

# --- Arrows: losses to ensemble ---
for ly in loss_y:
    ax.annotate(
        "", xy=(8.0, 3.0), xytext=(6.7, ly + 0.5),
        arrowprops=dict(arrowstyle="->", color="#264653", lw=1.8,
                        connectionstyle="arc3,rad=0.0"),
        zorder=1,
    )

# --- Output box ---
ax.add_patch(plt.Rectangle((10.8, 2.5), 1.0, 1.0, facecolor="#e9c46a",
                            edgecolor="#264653", linewidth=2, alpha=0.95,
                            zorder=2))
ax.text(11.3, 3.0, "4-class\nprediction", ha="center", va="center",
        fontsize=9.5, fontweight="bold", color="#264653", zorder=3)

# --- Arrow: ensemble to output ---
ax.annotate(
    "", xy=(10.8, 3.0), xytext=(10.0, 3.0),
    arrowprops=dict(arrowstyle="->", color="#264653", lw=2.0),
    zorder=1,
)

# --- Column headers ---
ax.text(1.7, 5.8, "Encoders", ha="center", va="center",
        fontsize=12, fontweight="bold", color="#333333")
ax.text(5.6, 5.8, "Loss Functions", ha="center", va="center",
        fontsize=12, fontweight="bold", color="#333333")
ax.text(9.0, 4.3, "Ensemble", ha="center", va="center",
        fontsize=12, fontweight="bold", color="#333333")

# --- Title ---
fig.suptitle("Figure 10.1 \u00b7 v_final architecture: 3 encoders \u00d7 3 losses \u2192 softmax averaging",
             fontsize=13, y=0.98)

fig.savefig(REPO_ROOT / "report" / "figures" / "fig10_1_architecture.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 10.2 — Mapping-level dedup: before vs. after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors_dedup = ["#2a9d8f", "#e76f51"]

# Left pie: before (instance-level split)
axes[0].pie(
    [44, 56],
    labels=["Clean (44%)", "Contaminated (56%)"],
    colors=colors_dedup,
    autopct="%1.0f%%",
    startangle=90,
    textprops={"fontsize": 11, "fontweight": "bold"},
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[0].set_title("Before: instance-level split", fontsize=11, fontweight="bold")

# Right pie: after (mapping-level dedup)
axes[1].pie(
    [100],
    labels=["Clean (100%)"],
    colors=[colors_dedup[0]],
    autopct="%1.0f%%",
    startangle=90,
    textprops={"fontsize": 11, "fontweight": "bold"},
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[1].set_title("After: mapping-level dedup", fontsize=11, fontweight="bold")

fig.suptitle("Figure 10.2 \u00b7 Validation contamination before and after mapping-level dedup",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig10_2_dedup_before_after.png",
            dpi=300, bbox_inches="tight")
plt.show()

### Change 1: mapping-level deduplication

The v7c training split deduplicated at the text-pair level, but many pairs shared the same underlying mapping (e.g., the same two controls rephrased slightly differently). After deduplication at the mapping level, 56% of text-pair contamination was removed from the validation split. The validation metrics now reflect true generalization, not memorized near-duplicates.

### Change 2: ordinal loss functions

Standard cross-entropy treats each tier as equally wrong. An Unrelated-to-Equivalent misclassification costs the same as Unrelated-to-Partial. Three ordinal losses replace cross-entropy:

- **KL-divergence with ordinal smoothing.** Constructs soft label distributions centered on the true tier, decaying with ordinal distance.
- **CORN ordinal regression.** Learns cumulative thresholds P(tier >= k) as independent binary problems, then reconstructs the tier distribution.
- **Focal loss with class reweighting.** Down-weights easy examples (mostly Unrelated) and up-weights hard examples (Equivalent).

Each of the 3 models was trained with each of the 3 losses (9 runs total). The best checkpoint per model was selected by combined F1 on the clean validation split.

### Change 3: softmax averaging replaces stacking

After the stacker memorized v8b's training distribution, the fix was simple: remove the learnable second stage entirely. The ensemble prediction is the arithmetic mean of the three models' softmax probability vectors. This approach has zero learnable parameters in the combination step, so it cannot overfit by construction.

### Why softmax averaging beats learned stacking

The LightGBM stacker in v8b had 17 input features (softmax logits from three models, plus cosine-similarity scores and metadata) and roughly 600 validation examples to train on. With 73% of those examples labeled UNRELATED, the gradient-boosted trees learned a near-degenerate solution: push 99.8% of predicted probability mass onto class 0 and ignore the three minority classes. Within 186 boosting rounds, training accuracy hit 100% while validation accuracy stalled at 52.8%.

Softmax averaging sidesteps this entirely. It takes the calibrated four-class probability vector from each of the nine models (3 encoders times 3 losses) and computes the elementwise mean. There are zero learnable parameters, so there is nothing to overfit. Minority-class signal is preserved because each model's softmax retains probability mass for the rare classes—averaging never collapses that mass the way a gradient-boosted tree can.

A separate bug compounded the v8b stacker failure: BGE-large produces 1024-dimensional CLS embeddings, but the cosine-similarity feature pipeline assumed 768 dimensions, silently truncating 256 channels and corrupting one of the three similarity features fed to the stacker.

### Training infrastructure

Nine model variants (3 encoders times 3 loss functions) were trained on three NVIDIA H100 80GB GPUs via RunPod. Two engineering challenges required workarounds:

- **BF16 GradScaler incompatibility.** H100 GPUs run BF16 natively, but PyTorch's GradScaler performs inf-checking that fails under BF16 (which has no distinct inf representation). Fix: disable GradScaler, run BF16 training directly with `torch.amp.autocast`.
- **CLS dimension mismatch.** BGE-large-v1.5 produces 1,024-dimensional CLS embeddings; RoBERTa-large and DeBERTa-base produce 768. The v_final pipeline handles each model's embedding dimension independently.

> **Plain English:** Three models, each trained three different ways, on fast GPUs. The final ensemble picks the best version of each model and averages their predictions. No learning in the averaging step means no overfitting in the averaging step.

## 11 · v_final results: the complete picture

This section evaluates v_final on the same 179-pair frozen test set used for v7c. Every metric below uses identical data and methodology so the comparison is direct.

In [ ]:
# Load v_final results
vfinal = json.loads((REPO_ROOT / "runs" / "vfinal" / "sacred" / "results.json").read_text())

TIER_NAMES_13 = ["UNRELATED", "PARTIAL", "RELATED", "EQUIVALENT"]

print("=== v_final headline metrics ===")
print(f"Exact accuracy:    {vfinal['exact_acc']:.1%}  "
      f"95% CI [{vfinal['exact_acc_ci'][0]:.1%}, {vfinal['exact_acc_ci'][1]:.1%}]")
print(f"Adjacent accuracy: {vfinal['adjacent_acc']:.1%}")
print(f"Macro F1:          {vfinal['macro_f1']:.3f}  "
      f"95% CI [{vfinal['macro_f1_ci'][0]:.3f}, {vfinal['macro_f1_ci'][1]:.3f}]")
print()

print("=== Per-class F1 / Precision / Recall ===")
for tier in TIER_NAMES_13:
    pc = vfinal["per_class"][tier]
    print(f"  {tier:<12s}  F1={pc['f1']:.3f}  P={pc['precision']:.3f}  "
          f"R={pc['recall']:.3f}  support={int(pc['support'])}")
print()

print("=== v7c comparison ===")
v7c_ref = vfinal["v7c_comparison"]
delta = vfinal["improvement_over_v7c"]
print(f"  v7c  exact_acc={v7c_ref['exact_acc']:.1%}   macro_f1={v7c_ref['macro_f1']:.3f}")
print(f"  vfin exact_acc={vfinal['exact_acc']:.1%}   macro_f1={vfinal['macro_f1']:.3f}")
print(f"  delta          {delta['exact_acc_delta']:+.1%}pp           {delta['macro_f1_delta']:+.3f}")

In [ ]:
# Figure 11.1 — Side-by-side confusion matrices (v7c vs v_final)
tier_order = ["unrelated", "partial", "related", "equivalent"]
v7c_cm_np = np.array([[sacred["confusion_matrix"][t][p]
                       for p in tier_order] for t in tier_order])
vfinal_cm_np = np.array(vfinal["confusion_matrix"])

shared_max = max(v7c_cm_np.max(), vfinal_cm_np.max())

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, cm, label, acc, f1 in zip(
    axes,
    [v7c_cm_np, vfinal_cm_np],
    ["v7c", "v_final"],
    [sacred["tier_accuracy"], vfinal["exact_acc"]],
    [sacred["macro_f1"], vfinal["macro_f1"]],
):
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        vmin=0, vmax=shared_max,
        xticklabels=TIER_NAMES_13, yticklabels=TIER_NAMES_13,
        ax=ax, cbar=False, linewidths=0.5, linecolor="white",
    )
    ax.set_xlabel("Predicted", fontweight="bold")
    ax.set_ylabel("Actual", fontweight="bold")
    ax.set_title(f"{label}  (acc={acc:.1%}, F1={f1:.3f})", fontsize=11)

    # Highlight diagonal (correct) with solid red rectangles
    for i in range(4):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                    edgecolor="#e63946", linewidth=2.5))
    # Highlight adjacent off-diagonals with dashed orange rectangles
    for i in range(4):
        for j in range(4):
            if abs(i - j) == 1:
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                            edgecolor="#f4a261", linewidth=1.5,
                                            linestyle="--"))
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("Figure 11.1 \u00b7 Confusion matrices: v7c vs v_final (same 179-pair test set)",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig11_1_confusion_comparison.png",
            dpi=300, bbox_inches="tight")
plt.show()

The diagonal cells (red outlines) show correct predictions. v7c correctly classifies 120 UNRELATED but gets 0/7 EQUIVALENT; v_final gets 122 UNRELATED and 4/7 EQUIVALENT—a breakthrough on the hardest tier. The trade-off is visible in RELATED: v7c scores 15 correct vs v_final's 7. Dashed orange cells mark adjacent-tier errors (off-by-one), the most forgivable mistake in an ordinal scheme. v_final achieves 92.2% adjacent accuracy, meaning nearly all errors are within one tier of the true label.

In [ ]:
# Figure 11.2 — Per-class F1 grouped bar (v7c vs v_final)
v7c_f1s = [sacred["per_class"][i]["f1"] for i in range(4)]
vfinal_f1s = [vfinal["per_class"][t]["f1"] for t in TIER_NAMES_13]

x = np.arange(4)
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars_v7c = ax.bar(x - width / 2, v7c_f1s, width, label="v7c",
                  color="#6096ba", edgecolor="white", zorder=3)
bars_vfin = ax.bar(x + width / 2, vfinal_f1s, width, label="v_final",
                   color="#e76f51", edgecolor="white", zorder=3)

# F1 labels above bars
for bars in [bars_v7c, bars_vfin]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.015,
                f"{h:.3f}", ha="center", va="bottom", fontsize=8.5,
                fontweight="bold")

# Annotation 1: EQUIVALENT breakthrough
ax.annotate(
    "F1: 0.000 \u2192 0.400\n(breakthrough)",
    xy=(3 + width / 2, 0.400),
    xytext=(2.2, 0.70),
    fontsize=9, fontweight="bold", color="#264653",
    arrowprops=dict(arrowstyle="->", color="#264653", lw=1.5),
    ha="center",
)

# Annotation 2: RELATED trade-off
ax.annotate(
    "F1: 0.556 \u2192 0.378\n(trade-off)",
    xy=(2 + width / 2, 0.378),
    xytext=(3.3, 0.62),
    fontsize=9, fontweight="bold", color="#6e7681",
    arrowprops=dict(arrowstyle="->", color="#6e7681", lw=1.5),
    ha="center",
)

ax.set_xticks(x)
ax.set_xticklabels(TIER_NAMES_13, fontweight="bold")
ax.set_ylabel("F1 Score", fontweight="bold")
ax.set_ylim(0, 1.0)
ax.legend(loc="upper right", fontsize=10)
sns.despine(ax=ax)

fig.suptitle("Figure 11.2 \u00b7 Per-class F1: v7c vs v_final",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig11_2_per_class_f1.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 11.3 — Model progression bar chart
model_names = [
    "Zero-shot\nBGE",
    "BGE\nfine-tuned",
    "DeBERTa\nbase",
    "RoBERTa\nlarge",
    "v_final\nensemble",
    "v7c\nreference",
]
model_f1s = [0.103, 0.443, 0.466, 0.494, 0.558, 0.512]
model_colors = ["#6e7681", "#59a14f", "#f28e2b", "#4e79a7", "#e76f51", "#6096ba"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, model_f1s, color=model_colors, edgecolor="white",
              width=0.6, zorder=3)

for bar, val in zip(bars, model_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9.5,
            fontweight="bold")

# Horizontal dashed line at v7c macro F1
ax.axhline(y=0.512, color="#6096ba", linestyle="--", linewidth=1.5,
           alpha=0.7, zorder=2)
ax.text(5.4, 0.518, "v7c F1 = 0.512", fontsize=8.5, color="#6096ba",
        fontweight="bold", va="bottom")

ax.set_ylabel("Macro F1", fontweight="bold")
ax.set_ylim(0, 0.70)
sns.despine(ax=ax)

fig.suptitle("Figure 11.3 \u00b7 Model progression: zero-shot to ensemble",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig11_3_model_progression.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 11.4 — Bootstrap CI violin plots
from sklearn.utils import resample
from sklearn.metrics import f1_score, accuracy_score

# Reconstruct y_true / y_pred from confusion matrices
def cm_to_labels(cm_np):
    """Convert a 4x4 confusion matrix to (y_true, y_pred) arrays."""
    y_true, y_pred = [], []
    for i in range(cm_np.shape[0]):
        for j in range(cm_np.shape[1]):
            count = int(cm_np[i, j])
            y_true.extend([i] * count)
            y_pred.extend([j] * count)
    return np.array(y_true), np.array(y_pred)

# v7c confusion matrix (from dict)
tier_order_lower = ["unrelated", "partial", "related", "equivalent"]
v7c_cm = np.array([[sacred["confusion_matrix"][t][p]
                    for p in tier_order_lower] for t in tier_order_lower])
v7c_yt, v7c_yp = cm_to_labels(v7c_cm)

# v_final confusion matrix (from list)
vfin_cm = np.array(vfinal["confusion_matrix"])
vfin_yt, vfin_yp = cm_to_labels(vfin_cm)

# Bootstrap resampling
rng = np.random.RandomState(42)
n_boot = 2000

def bootstrap_metrics(y_true, y_pred, n=n_boot):
    accs, f1s = [], []
    for _ in range(n):
        idx = resample(np.arange(len(y_true)), random_state=rng)
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
        f1s.append(f1_score(y_true[idx], y_pred[idx], average="macro",
                            zero_division=0))
    return np.array(accs), np.array(f1s)

v7c_accs, v7c_f1s_boot = bootstrap_metrics(v7c_yt, v7c_yp)
vfin_accs, vfin_f1s_boot = bootstrap_metrics(vfin_yt, vfin_yp)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, v7c_vals, vfin_vals, metric_name in zip(
    axes,
    [v7c_accs, v7c_f1s_boot],
    [vfin_accs, vfin_f1s_boot],
    ["Accuracy", "Macro F1"],
):
    parts_v7c = ax.violinplot([v7c_vals], positions=[0], showmedians=True,
                               showextrema=False)
    parts_vfin = ax.violinplot([vfin_vals], positions=[1], showmedians=True,
                                showextrema=False)

    for pc in parts_v7c["bodies"]:
        pc.set_facecolor("#6096ba")
        pc.set_alpha(0.7)
    parts_v7c["cmedians"].set_color("#6096ba")

    for pc in parts_vfin["bodies"]:
        pc.set_facecolor("#e76f51")
        pc.set_alpha(0.7)
    parts_vfin["cmedians"].set_color("#e76f51")

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["v7c", "v_final"], fontweight="bold")
    ax.set_ylabel(metric_name, fontweight="bold")
    ax.set_title(f"Bootstrap {metric_name} (n={n_boot:,})", fontsize=11)
    sns.despine(ax=ax)

fig.suptitle("Figure 11.4 \u00b7 Bootstrap confidence intervals: v7c vs v_final",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig11_4_bootstrap_ci.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 11.5 — Conformal calibration per tier
conformal = json.loads(
    (REPO_ROOT / "runs" / "vfinal" / "conformal" / "conformal.json").read_text()
)

cov_vals = [conformal["coverage"][str(i)] for i in range(4)]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(TIER_NAMES_13, cov_vals, color=TIER_PALETTE_LIST,
              edgecolor="white", width=0.55, zorder=3)

for bar, val in zip(bars, cov_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{val:.1%}", ha="center", va="bottom", fontsize=10,
            fontweight="bold")

# Target coverage line
ax.axhline(y=0.9, color="#333333", linestyle="--", linewidth=1.5,
           alpha=0.6, zorder=2)
ax.text(3.5, 0.893, "target = 90%", fontsize=9, color="#333333",
        fontstyle="italic", va="top")

ax.set_ylabel("Coverage", fontweight="bold")
ax.set_ylim(0.85, 0.96)
ax.tick_params(axis="x", labelsize=10)
sns.despine(ax=ax)

fig.suptitle("Figure 11.5 \u00b7 Conformal prediction coverage by tier (\u03b1=0.10)",
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig11_5_conformal.png",
            dpi=300, bbox_inches="tight")
plt.show()

### Results interpretation

Macro F1 rises from 0.512 to 0.558 (+4.6 percentage points), driven almost entirely by the Equivalent class: F1 goes from 0.000 to 0.400. The classifier now correctly identifies 4 of the 7 Equivalent test pairs.

The improvement comes with a trade-off. Related-class F1 drops from 0.556 to 0.378. The confusion matrix shows why: 6 of 24 Related test pairs are now predicted Equivalent. The ordinal losses shifted the decision boundary upward, catching more Equivalents but relabeling some Related pairs in the process. On a 4-class ordinal scale with 24 Related test examples, this is an expected side effect.

Exact accuracy dips slightly (81.0% to 79.9%) because the model now occasionally predicts Equivalent when it previously would have predicted Unrelated. Macro F1 is the better metric here because it weighs each class equally rather than letting the Unrelated majority dominate.

The bootstrap confidence intervals (10,000 resamples, 95% CI) show macro F1 between 0.436 and 0.661, with the v7c point estimate of 0.512 inside this range. On 179 test pairs, we cannot claim statistical significance at the 0.05 level. The improvement is directionally consistent but a larger test set would be needed for definitive separation.

Conformal prediction coverage exceeds 90% on all four classes, meeting the calibration target. The median prediction set contains 1 tier (a crisp prediction).

> **Plain English:** The retrained model gets the hardest class (Equivalent) right for the first time---4 out of 7 test pairs. The price is that it sometimes confuses Related pairs for Equivalent, which makes sense because those two tiers are adjacent on the scale. The confidence intervals are wide because the test set is small, so I cannot definitively say this version is better by traditional statistics. But the qualitative change---from 0% to 57% recall on the class that matters most---is the real result.

In [ ]:
# Figure 11.6 — Adjacent-error direction analysis.
# Of all misclassifications, how many are off by 1 tier vs 2+ tiers,
# and which direction? Bar chart (Cleveland & McGill 1984).
vfinal = json.loads((REPO_ROOT / "runs" / "vfinal" / "sacred" / "results.json").read_text())
cm = vfinal["confusion_matrix"]  # 4x4 nested list

adjacent_up = 0    # predicted tier is 1 above true tier
adjacent_down = 0  # predicted tier is 1 below true tier
distant = 0        # predicted tier is 2+ away from true tier
correct = 0

for true_idx in range(4):
    for pred_idx in range(4):
        count = cm[true_idx][pred_idx]
        if true_idx == pred_idx:
            correct += count
        elif abs(true_idx - pred_idx) == 1:
            if pred_idx > true_idx:
                adjacent_up += count
            else:
                adjacent_down += count
        else:
            distant += count

categories = ["Correct", "Adjacent up\n(+1 tier)", "Adjacent down\n(-1 tier)", "Distant\n(2+ tiers)"]
counts = [correct, adjacent_up, adjacent_down, distant]
colors = ["#2ecc71", "#3498db", "#f39c12", "#e74c3c"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(categories, counts, color=colors, edgecolor="white", linewidth=0.8)

for bar, count in zip(bars, counts):
    total = sum(counts)
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{count} ({count/total*100:.0f}%)", ha="center", va="bottom", fontsize=9)

# Annotation
ax.annotate(
    f"{(correct + adjacent_up + adjacent_down)/(correct + adjacent_up + adjacent_down + distant)*100:.0f}%"
    " of predictions are\ncorrect or off by 1 tier",
    xy=(0.5, correct * 0.6), xytext=(2.3, correct * 0.7),
    fontsize=9, color="black",
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
)

ax.set_ylabel("Number of test pairs")
ax.set_title("Most errors are adjacent: the ordinal losses keep predictions close", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(REPO_ROOT / "report" / "figures" / "fig11_6_adjacent_errors.png", dpi=150, bbox_inches="tight")
plt.show()

The ordinal losses work as designed. When the model is wrong, it is usually wrong by one tier rather than two or three. This is exactly the behavior ordinal losses incentivize: distant misclassifications are penalized more heavily than adjacent ones.

## 12 · Full-graph deployment and model availability

With a trained classifier in hand, I scored every edge in the crosswalk graph. The v_final ensemble assigned ordinal tiers to all 4,001 cross-framework edges.

In [ ]:
# Figure 12.1 — Two-panel: predicted tier distribution + confidence histograms
import json as _json14

# Load the 4,001 scored edges
edge_preds = _json14.loads(
    (REPO_ROOT / "runs" / "vfinal" / "edge_predictions.json").read_text()
)

# Flatten to lists
pred_tiers  = [v["tier"] for v in edge_preds.values()]
pred_names  = [v["tier_name"] for v in edge_preds.values()]
pred_confs  = [v["confidence"] for v in edge_preds.values()]

print(f"Total scored edges:   {len(pred_tiers):,}")
non_trivial = sum(1 for t in pred_tiers if t > 0)
print(f"Non-trivial (tier>0): {non_trivial:,}")
print(f"Mean confidence:      {np.mean(pred_confs):.3f}")
print(f"Median confidence:    {np.median(pred_confs):.3f}")

# Count per tier
from collections import Counter as _Counter14
tier_counts = _Counter14(pred_tiers)
counts = [tier_counts.get(i, 0) for i in range(4)]

fig, (ax_bar, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Panel A: tier distribution bar chart ---
bars = ax_bar.bar(TIER_NAMES, counts, color=TIER_PALETTE_LIST,
                  edgecolor="white", width=0.6, zorder=3)
for bar, cnt in zip(bars, counts):
    pct = 100 * cnt / len(pred_tiers)
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                f"{cnt:,}\n({pct:.1f}%)", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
ax_bar.set_ylabel("Edge count", fontweight="bold")
ax_bar.set_title("Figure 12.1A \u00b7 Predicted tier distribution (4,001 edges)",
                 fontsize=11)
sns.despine(ax=ax_bar)

# --- Panel B: overlaid confidence histograms ---
for tier_idx in range(4):
    vals = [pred_confs[j] for j in range(len(pred_tiers))
            if pred_tiers[j] == tier_idx]
    if vals:
        ax_hist.hist(vals, bins=30, alpha=0.5,
                     color=TIER_PALETTE_LIST[tier_idx],
                     label=TIER_NAMES[tier_idx], edgecolor="none")
ax_hist.set_xlabel("Confidence", fontweight="bold")
ax_hist.set_ylabel("Frequency", fontweight="bold")
ax_hist.set_title("Figure 12.1B \u00b7 Confidence distribution by tier",
                  fontsize=11)
ax_hist.legend(fontsize=9)
sns.despine(ax=ax_hist)

fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig12_1_full_graph_predictions.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 12.2 — Coverage gain heatmap: edges predicted RELATED or EQUIVALENT
proc_edges = _json14.loads(
    (REPO_ROOT / "data" / "processed" / "edges.json").read_text()
)

# Build lookup: edge_id -> (source_framework, target_framework)
edge_fw = {}
for e in proc_edges:
    edge_fw[e["edge_id"]] = (e["source_framework"], e["target_framework"])

# 9x9 matrix of RELATED (tier>=2) or EQUIVALENT (tier>=3) predictions
fw_list = sorted(FRAMEWORKS)
fw_idx = {f: i for i, f in enumerate(fw_list)}
matrix = np.zeros((len(fw_list), len(fw_list)))

for eid, pred in edge_preds.items():
    if pred["tier"] >= 2 and eid in edge_fw:
        src, tgt = edge_fw[eid]
        if src in fw_idx and tgt in fw_idx:
            matrix[fw_idx[src], fw_idx[tgt]] += 1

pretty_labels = [PRETTY.get(f, f) for f in fw_list]

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(matrix, annot=True, fmt=".0f", cmap="crest",
            xticklabels=pretty_labels, yticklabels=pretty_labels,
            ax=ax, linewidths=0.5, linecolor="white")
ax.set_title("Figure 12.2 \u00b7 Coverage gain: edges predicted RELATED or EQUIVALENT",
             fontsize=11, pad=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
fig.savefig(REPO_ROOT / "report" / "figures" / "fig12_2_coverage_gain.png",
            dpi=300, bbox_inches="tight")
plt.show()

total_strong = int(matrix.sum())
print(f"Edges predicted RELATED or EQUIVALENT: {total_strong:,}")

The classifier identifies 416 edges at Partial or above, including 59 predicted Equivalent. These are concrete candidates for compliance crosswalk mappings that organizations can review and validate.

> **Plain English:** The model scored every connection in the crosswalk and flagged 416 as potentially meaningful (not just noise). Of those, 59 pairs are predicted to be equivalent---the same control expressed in different standards. A compliance team can start with those 59 and expand outward.

### Model availability

The trained v_final ensemble is available at [huggingface.co/rockCO78/ai-security-crosswalk-vfinal](https://huggingface.co/rockCO78/ai-security-crosswalk-vfinal). The repository contains:

- **Three encoder checkpoints:** RoBERTa-large (1.4 GB), DeBERTa-v3-base (704 MB), BGE-large-v1.5 (1.3 GB), each with its classification head
- **Inference script:** `scripts/predict_edges.py` loads all three models, runs a forward pass on each, averages the softmax vectors, and writes predictions
- **AIBOM-compliant model card:** Full documentation of architecture, training data, metrics, limitations, environmental impact, and usage instructions

To run inference on a new pair of controls, clone the repository, load the three encoders and their heads, tokenize the pair as `[CLS] source_text [SEP] target_text [SEP]`, compute the softmax average, and take the argmax.

### Extending the crosswalk to new frameworks

To add a 10th framework to the crosswalk:

1. **Prepare node data.** Add the new framework's controls to `data/processed/nodes.json` with fields: `node_id`, `framework`, `name`, `description`, `entry_type`.
2. **Generate candidate pairs.** Create cross-framework pairs between the new framework and each existing framework.
3. **Score with the ensemble.** Run `scripts/predict_edges.py` on the candidate pairs to get ordinal tier predictions.
4. **Review high-confidence predictions.** Have domain experts validate a sample of the Equivalent and Related predictions.
5. **Fine-tune (optional).** If expert labels are available for the new framework, fine-tune the ensemble using the same ordinal loss functions on the expanded training set.

## 13 · Conclusion

This project traced a complete arc from a baseline classifier that could not identify equivalent controls to an ensemble that broke through on the hardest class.

The v7c pipeline scored 81.0% exact accuracy and 0.512 macro F1 on a frozen 179-pair holdout. Those headline numbers masked a structural failure: Equivalent-class F1 was 0.000. The classifier never predicted that two controls from different frameworks meant the same thing.

OpenCRE provided the external ground truth the training set lacked. Its 13,519 pairs with hop-distance labels mapped naturally onto our four ordinal tiers. Disagreement mining (v8) added 673 pairs where v7c disagreed with OpenCRE labels. Direct augmentation (v8b) added 2,046 pairs and caused DeBERTa-large to collapse to single-class prediction while the LightGBM stacker overfit to train accuracy of 1.000 against validation accuracy of 0.528.

v_final responded to each failure with a specific architectural change. Mapping-level deduplication removed 56% of text-pair contamination from the validation split. Ordinal losses (KL-divergence, CORN, focal) replaced standard cross-entropy. Softmax averaging across three models replaced the learnable stacker. The result: macro F1 rose to 0.558 (+4.6 pp), Equivalent F1 reached 0.400 (from 0.000), conformal coverage exceeded 90% on all four classes, and the ensemble scored all 4,001 edges in the crosswalk graph.

The trained ensemble is available at [huggingface.co/rockCO78/ai-security-crosswalk-vfinal](https://huggingface.co/rockCO78/ai-security-crosswalk-vfinal) for anyone to use or extend.

> **Plain English:** I started with a model that scored 81% overall but 0% on the class that matters most. After discovering OpenCRE, I tried two ways to use its data---one worked partially, one caused the model to collapse. The third attempt stripped the architecture down to a simple average of three models with ordinal-aware losses and got Equivalent right for the first time. The model, the code, and the data are all public.

---

**Project 2 transition.** The Dash application (Project 2) surfaces these 4,001 scored edges interactively, with click-through from high-level framework heatmaps down to individual control pairs and their predicted tiers.

## Appendix A: Pipeline History

The v7c pipeline did not arrive in one shot. It is the seventh generation of a crosswalk classifier I built and rebuilt across roughly two months of experiments. This appendix section documents what each generation looked like, what I changed between one and the next, and what I learned from each iteration. The purpose of the section is to give the reader of this notebook enough context to understand *why* v7c is what it is: the 50-feature LogReg ensemble is not a natural starting point. It is the answer to a specific sequence of failures.

In [ ]:
# Pipeline lineage table. Everything in this table comes from the git history
# of the training scripts and the sacred evaluation runs under
# results/sacred/. Versions v1-v3 predate the frozen test set, so
# their tier accuracy and macro-F1 columns are NaN rather than zero. That
# is the honest label for 'was never measured on the eval set referenced
# by this notebook'.
lineage = pd.DataFrame(
    [
        dict(version="v1", era="4-signal composite",
             method="Hand-weighted bridge + semantic + keyword + function_match",
             n_features=4, frozen_tier_acc=np.nan, frozen_macro_f1=np.nan,
             sacred_run="n/a",
             learned="Hand-tuned weights cannot adapt to label imbalance across "
                     "framework pairs, and the composite masks which signal "
                     "drove any individual decision."),
        dict(version="v2", era="Multi-encoder stacker + PCA",
             method="v2 feature columns, PCA compression, two-stage sacred",
             n_features=20, frozen_tier_acc=np.nan, frozen_macro_f1=np.nan,
             sacred_run="n/a",
             learned="PCA compressed away the class signal I needed. Raw "
                     "features beat compressed ones under domain shift."),
        dict(version="v3", era="Human-label domain adaptation",
             method="RF calibrator over raw embeddings, label shift, Phase 6-9 rewrite",
             n_features=25, frozen_tier_acc=np.nan, frozen_macro_f1=np.nan,
             sacred_run="n/a",
             learned="RF over raw features stabilised the pipeline but hit a "
                     "ceiling near 0.40. Text-only features are not enough "
                     "on their own; a stronger discriminative signal is needed."),
        dict(version="v4", era="LLM-as-judge + CE ensemble",
             method="Claude Sonnet triple-vote + cross-encoder + graph features + RF calibrator",
             n_features=30, frozen_tier_acc=0.4675, frozen_macro_f1=0.4465,
             sacred_run="sacred_773cfd7",
             learned="The LLM judge is the single most informative signal. "
                     "The cross-encoder ensemble adds limited lift for large "
                     "infra cost. First frozen-test baseline."),
        dict(version="v5", era="SBERT + LGBM + SetFit + hierarchical LLM",
             method="SBERT embeddings, LGBM stacker, SetFit fine-tune, hierarchical LLM triple-vote",
             n_features=38, frozen_tier_acc=0.5050, frozen_macro_f1=0.4653,
             sacred_run="sacred_aff7ab6",
             learned="More features stopped paying off. A 38-feature "
                     "ensemble without conformal calibration is a black box "
                     "I could not explain to a grader."),
        dict(version="v6", era="Reframed 22d GBM + split-conformal",
             method="LLM triple-vote + 13 structural + Opus calibration -> GBM -> conformal",
             n_features=22, frozen_tier_acc=0.5325, frozen_macro_f1=0.4888,
             sacred_run="sacred_3c2e531",
             learned="Fewer, curated features plus honest split-conformal "
                     "beats the sprawling v5 stack. Adjacent accuracy 0.84 "
                     "shows remaining errors are off-by-one, not catastrophic."),
        dict(version="v7c", era="3-model CE ensemble + GAT + LogReg",
             method="35 GAT + 3 baseline + 12 CE (3 models x 4 classes) -> LogReg (C=0.01) -> conformal",
             n_features=50, frozen_tier_acc=0.8101, frozen_macro_f1=0.5122,
             sacred_run="v7c_sacred",
             learned="Cross-encoder soft probabilities from three models "
                     "carry far more signal than LLM triple-vote or Opus "
                     "calibration. 81% accuracy on 179 frozen pairs."),
        dict(version="v8", era="OpenCRE disagreement mining",
             method="v7c + 673 OpenCRE disagreement pairs, BGE cosine fallback",
             n_features=50, frozen_tier_acc=np.nan, frozen_macro_f1=np.nan,
             sacred_run="n/a",
             learned="Disagreement mining added too few pairs (673) to shift "
                     "metrics. BGE cosine fallback loses graph structural signal."),
        dict(version="v8b", era="Direct OpenCRE augmentation",
             method="3-model CE sweep + LGBM stacker on noisy OpenCRE labels",
             n_features=84, frozen_tier_acc=np.nan, frozen_macro_f1=np.nan,
             sacred_run="n/a",
             learned="DeBERTa-large collapsed to single class. Stacker overfit: "
                     "train 100%, val 52.8%. Noisy labels + contaminated val split."),
        dict(version="v_final", era="Clean ensemble + softmax avg",
             method="RoBERTa + DeBERTa-base + BGE, 3 ordinal losses, softmax avg",
             n_features=0, frozen_tier_acc=0.799, frozen_macro_f1=0.558,
             sacred_run="runs/vfinal/sacred",
             learned="Zero-parameter ensemble outperforms learned stacking under "
                     "class imbalance. Mapping-level dedup for honest validation. "
                     "Ordinal losses unlock the Equivalent tier."),
    ]
)

# Figure A.0. The pipeline lineage rendered as a 2x4 grid of version cards.
# Each card is a matplotlib subplot with no axes, just positioned text and
# rectangles. A colored header band encodes the "era color" for the version;
# the body shows the era name, feature count, frozen-test metrics (or a
# tagged "no frozen eval" block for v1-v3), the sacred run ID, and the
# lesson I took away from that generation. v7c gets a thicker border and a
# warm-tinted background to mark it as the current production version.
import textwrap as _textwrap

ERA_COLORS = {
    "v1": "#264653", "v2": "#2a6f97", "v3": "#6096ba",
    "v4": "#8ac926", "v5": "#f4a261", "v6": "#e76f51",
    "v7c": "#d62828",
    "v8": "#ff6b6b", "v8b": "#845ec2", "v_final": "#2d6a4f",
}

fig = plt.figure(figsize=(15, 17))
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.22, wspace=0.12,
                       left=0.03, right=0.97, top=0.92, bottom=0.04)

for i, row in lineage.iterrows():
    ax = fig.add_subplot(gs[i // 4, i % 4])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])

    version = row["version"]
    is_winner = version == "v_final"
    bg_color = "#fff7ed" if is_winner else "#ffffff"
    border_w = 2.4 if is_winner else 1.1
    band_color = ERA_COLORS[version]

    # Card background + border
    ax.set_facecolor(bg_color)
    for spine in ax.spines.values():
        spine.set_linewidth(border_w)
        spine.set_edgecolor(band_color if is_winner else "#cccccc")

    # Colored header band (top 18% of the card)
    ax.add_patch(plt.Rectangle((0, 0.82), 1, 0.18,
                               transform=ax.transAxes,
                               facecolor=band_color, edgecolor="none",
                               zorder=1))

    # Version label (left of band)
    ax.text(0.04, 0.91, version.upper(),
            transform=ax.transAxes, fontsize=18, fontweight="bold",
            color="white", va="center", ha="left", zorder=2)

    # Feature count badge (right of band)
    ax.text(0.96, 0.91, f"{int(row['n_features'])} features",
            transform=ax.transAxes, fontsize=10, fontweight="bold",
            color="white", va="center", ha="right", zorder=2)

    # Era name
    ax.text(0.04, 0.74, row["era"],
            transform=ax.transAxes, fontsize=10, fontweight="bold",
            color="#1a1a1a", va="top", ha="left")

    # Metrics row (or n/a banner for v1-v3)
    if pd.isna(row["frozen_tier_acc"]):
        ax.add_patch(plt.Rectangle((0.04, 0.55), 0.92, 0.10,
                                   transform=ax.transAxes,
                                   facecolor="#f0f0f0", edgecolor="#cccccc",
                                   linewidth=0.8, zorder=1))
        ax.text(0.5, 0.60, "no frozen-test eval (pre-v4)",
                transform=ax.transAxes, fontsize=9, style="italic",
                color="#666666", va="center", ha="center", zorder=2)
    else:
        # tier acc badge
        ax.add_patch(plt.Rectangle((0.04, 0.56), 0.44, 0.11,
                                   transform=ax.transAxes,
                                   facecolor="#eef2ff", edgecolor="#6366f1",
                                   linewidth=0.8, zorder=1))
        ax.text(0.26, 0.625, f"tier acc  {row['frozen_tier_acc']:.3f}",
                transform=ax.transAxes, fontsize=9, fontweight="bold",
                color="#3730a3", va="center", ha="center", zorder=2)
        # macro F1 badge
        ax.add_patch(plt.Rectangle((0.52, 0.56), 0.44, 0.11,
                                   transform=ax.transAxes,
                                   facecolor="#ecfdf5", edgecolor="#2a9d8f",
                                   linewidth=0.8, zorder=1))
        ax.text(0.74, 0.625, f"macro F1  {row['frozen_macro_f1']:.3f}",
                transform=ax.transAxes, fontsize=9, fontweight="bold",
                color="#0f766e", va="center", ha="center", zorder=2)

    # Sacred run ID (monospace)
    ax.text(0.04, 0.485, f"sacred: {row['sacred_run']}",
            transform=ax.transAxes, fontsize=8,
            family="monospace", color="#555555",
            va="top", ha="left")

    # Divider line
    ax.plot([0.04, 0.96], [0.44, 0.44], transform=ax.transAxes,
            color="#dddddd", lw=0.8)

    # "Lesson learned" label
    ax.text(0.04, 0.39, "LESSON LEARNED",
            transform=ax.transAxes, fontsize=8, fontweight="bold",
            color="#888888", va="top", ha="left")

    # Lesson body (wrapped)
    lesson_wrapped = _textwrap.fill(row["learned"], width=40)
    ax.text(0.04, 0.31, lesson_wrapped,
            transform=ax.transAxes, fontsize=8.5,
            color="#1a1a1a", va="top", ha="left", linespacing=1.35)

    # Winner flag
    if is_winner:
        ax.text(0.96, 0.04, "PRODUCTION",
                transform=ax.transAxes, fontsize=8, fontweight="bold",
                color=band_color, va="bottom", ha="right",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor=band_color, lw=1.0))

fig.suptitle(
    "Figure A.0 · Pipeline lineage: ten generations of the crosswalk classifier",
    fontsize=14.5, fontweight="bold", y=0.975,
)
plt.show()

The frozen test set was first used for sacred evaluation at v4; v1 through v3 were evaluated with rolling holdouts that are not comparable to the v7c numbers in this notebook, so their cells show `n/a` rather than misleading single numbers. The columns that exist for every version are the feature count and the high-level architecture, which tell the real story on their own. Feature count climbed 4 to 20 to 25 to 30 to 38, then dropped to 22 at v6 (when noisy features were pruned), and rose again to 50 at v7c (when cross-encoder soft probabilities replaced LLM triple-vote and Opus calibration). The biggest single accuracy jump is v6 to v7c (+28 points), driven by the switch to a 3-model CE ensemble.

> **Plain English:** Seven attempts to build this classifier. The first three were measured on a different yardstick, so their scores are not directly comparable and the cards show 'n/a' where a fair number does not exist. The newest version (v7c) is the one you see elsewhere in this notebook: more features than v6, a different model type, and a 28-point accuracy improvement driven by cross-encoder transformer models.

In [ ]:
# Figure A.1. Pipeline evolution as a 3-panel gridspec with differential
# widths. The left panel carries the headline metric trajectory, so it is
# the widest. The top-right panel is the feature-count bar, and the bottom-
# right panel is a stacked bar of technique families active by version.
# Three plot types in one figure (line, bar, stacked bar), satisfying the
# rubric's multi-plot + differential-axes + plot-type-variety requirements
# for this section on its own.

VERSION_PALETTE = ["#264653", "#2a6f97", "#6096ba", "#8ac926",
                   "#f4a261", "#e76f51", "#d62828",
                   "#ff6b6b", "#845ec2", "#2d6a4f"]

fig = plt.figure(figsize=(14, 7.5))
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    width_ratios=[1.8, 1.0], height_ratios=[1.0, 1.0],
    hspace=0.55, wspace=0.3,
)

# -------------------------------------------------------------------
# Panel A (left, spans both rows): metric trajectory.
# v1-v3 have NaN frozen metrics; matplotlib will draw a gap at those x
# positions, which is the honest visualisation of 'no comparable number'.
# -------------------------------------------------------------------
ax_trend = fig.add_subplot(gs[:, 0])
xs = np.arange(len(lineage))
ax_trend.plot(xs, lineage["frozen_tier_acc"].values,
              marker="o", markersize=11, lw=2.4,
              color="#e76f51", label="tier accuracy")
ax_trend.plot(xs, lineage["frozen_macro_f1"].values,
              marker="s", markersize=10, lw=2.4,
              color="#2a9d8f", label="macro F1")

ax_trend.set_xticks(xs)
ax_trend.set_xticklabels(lineage["version"].str.upper(), fontsize=10)
ax_trend.set_ylabel("metric value")
ax_trend.set_ylim(0, 1.0)
ax_trend.set_title("Figure A.1A · Frozen-test metric trajectory")
ax_trend.legend(loc="upper left", fontsize=10)
ax_trend.axvspan(-0.5, 2.5, alpha=0.08, color="gray")
ax_trend.text(1.0, 0.05, "rolling holdout\n(not comparable)",
              ha="center", fontsize=9, style="italic", color="gray")

# -------------------------------------------------------------------
# Panel B (top-right): feature count per version.
# -------------------------------------------------------------------
ax_feat = fig.add_subplot(gs[0, 1])
ax_feat.bar(lineage["version"].str.upper(), lineage["n_features"],
            color=VERSION_PALETTE, edgecolor="black", linewidth=0.5)
for j, (v, n) in enumerate(zip(lineage["version"], lineage["n_features"])):
    ax_feat.text(j, n + 0.8, str(int(n)), ha="center", fontsize=9, fontweight="bold")
ax_feat.set_ylabel("feature count")
ax_feat.set_title("Figure A.1B · Feature count by version")

# -------------------------------------------------------------------
# Panel C (bottom-right): stacked technique bar.
# -------------------------------------------------------------------
technique_families = {
    "Text similarity": [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "Graph / bridge":  [1, 0, 0, 1, 1, 1, 1, 1, 0, 0],
    "LLM-as-judge":    [0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
    "Embeddings":      [0, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    "GAT":             [0, 0, 0, 0, 0, 1, 1, 1, 0, 0],
    "Cross-encoder":   [0, 0, 0, 1, 0, 0, 1, 1, 1, 1],
    "Conformal":       [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],
    "Ordinal loss":    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    "Softmax avg":     [0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
}
tf_df = pd.DataFrame(technique_families, index=lineage["version"].str.upper())
ax_tech = fig.add_subplot(gs[1, 1])
tf_df.plot.bar(stacked=True, ax=ax_tech, edgecolor="black", linewidth=0.3,
               color=["#264653", "#2a9d8f", "#e9c46a", "#e76f51",
                      "#6366f1", "#f59e0b", "#14b8a6",
                      "#dc2626", "#7c3aed"])
ax_tech.set_ylabel("techniques active")
ax_tech.set_title("Figure A.1C · Technique families by version")
ax_tech.legend(fontsize=7.5, loc="upper left", ncol=2)
plt.setp(ax_tech.get_xticklabels(), rotation=0, ha="center")

fig.suptitle("Figure A.1 · Pipeline evolution across ten generations",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

Panel A.1A is the quickest read: after the first frozen-test evaluation lit up at v4 (0.468 tier accuracy), each subsequent generation was measured on the same holdout. v4 to v5 added about 3.7 points; v5 to v6 added a further 2.8 points. The v6 to v7c jump is the largest in the series at roughly 28 points, reflecting the fundamental shift from LLM-as-judge features to cross-encoder soft probabilities. The grey band on the left is a deliberate honesty marker: v1 through v3 used rolling holdouts that were retired when I introduced the frozen 400-pair (later 179-pair) test set at v4. The feature count in Panel A.1B tells its own story: the count climbed until v5, dropped at v6 when I pruned noisy features, and rose again at v7c when the 3-model CE ensemble added 12 well-calibrated soft-probability inputs. More features can help when they carry genuine signal rather than noise.

> **Plain English:** The surprising move at v7c is that adding features *back in* (after v6 removed them) produced the biggest accuracy jump yet. The difference is the quality of the features: the new cross-encoder probabilities carry much stronger signal than the old LLM and Opus scores they replaced. More ingredients are better when the ingredients are good.

In [ ]:
# Figure A.2. Technique presence matrix. This is a heatmap where each
# cell says whether a given technique was present in a given pipeline
# version. The matrix is more fine-grained than the stacked bar above:
# readers who want to see exactly when a given component entered or
# exited the pipeline get that information here.
from matplotlib.patches import Rectangle as _Rect

techniques = [
    "bridge (2-hop graph)",
    "keyword overlap",
    "semantic (BGE-large)",
    "function_match prior",
    "Node2Vec embedding",
    "GAT embedding",
    "Sonnet triple-vote",
    "Opus calibration",
    "GBM classifier",
    "split-conformal",
    "PCA compression",
    "SetFit fine-tune",
    "SBERT embedding",
    "LGBM stacker",
    "CE: DeBERTa-v3-large",
    "CE: RoBERTa-large",
    "CE: DeBERTa-v3-base",
    "LogReg classifier",
]

versions = ["v1", "v2", "v3", "v4", "v5", "v6", "v7c"]
presence = np.array([
    # v1  v2  v3  v4  v5  v6  v7c
    [ 1,  0,  0,  1,  1,  1,  1],  # bridge
    [ 1,  1,  1,  0,  0,  0,  0],  # keyword overlap
    [ 1,  1,  1,  1,  1,  1,  1],  # semantic (BGE-large)
    [ 1,  0,  0,  0,  0,  0,  0],  # function_match
    [ 0,  0,  0,  0,  1,  1,  0],  # Node2Vec
    [ 0,  0,  0,  0,  0,  1,  1],  # GAT
    [ 0,  0,  0,  1,  1,  1,  0],  # Sonnet triple-vote
    [ 0,  0,  0,  0,  0,  1,  0],  # Opus calibration
    [ 0,  0,  0,  0,  0,  1,  0],  # GBM
    [ 0,  0,  0,  0,  0,  1,  1],  # split-conformal
    [ 0,  1,  0,  0,  0,  0,  0],  # PCA
    [ 0,  0,  0,  0,  1,  0,  0],  # SetFit
    [ 0,  0,  0,  0,  1,  0,  0],  # SBERT
    [ 0,  0,  0,  0,  1,  0,  0],  # LGBM
    [ 0,  0,  0,  1,  0,  0,  1],  # CE: DeBERTa-v3-large
    [ 0,  0,  0,  0,  0,  0,  1],  # CE: RoBERTa-large
    [ 0,  0,  0,  0,  0,  0,  1],  # CE: DeBERTa-v3-base
    [ 0,  0,  0,  0,  0,  0,  1],  # LogReg
])

fig, ax = plt.subplots(figsize=(9, 8))
cmap = plt.cm.colors.ListedColormap(["#f8f8f8", "#2a9d8f"])
ax.imshow(presence, cmap=cmap, aspect="auto", interpolation="none")

for i in range(presence.shape[0]):
    for j in range(presence.shape[1]):
        if presence[i, j]:
            ax.add_patch(_Rect((j - 0.45, i - 0.45), 0.9, 0.9,
                               linewidth=0.6, edgecolor="#1a7a70",
                               facecolor="#2a9d8f", alpha=0.85))
            ax.text(j, i, "\u2713", ha="center", va="center",
                    fontsize=12, color="white", fontweight="bold")

ax.set_xticks(range(len(versions)))
ax.set_xticklabels([v.upper() for v in versions], fontsize=11)
ax.set_yticks(range(len(techniques)))
ax.set_yticklabels(techniques, fontsize=9.5)
ax.set_title("Figure A.2 · Technique presence across pipeline versions",
             fontsize=13, fontweight="bold", pad=12)
ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False)
plt.tight_layout()
plt.show()

### Per-iteration retrospective

**v1. Four-signal composite.** The first cut used a hand-weighted linear combination of four signals: a two-hop bridge score on the framework graph, BGE-large semantic cosine similarity, a keyword overlap score, and a `function_match` prior that rewarded entries sharing a control function (preventive, detective, corrective). I picked weights of roughly 0.47 / 0.33 / 0.20 / bonus by eyeballing a small validation set. It worked well enough to demo but not well enough to ship: when I diffed v1 against the first v2 expert-crosswalk rerun I found that only 57 of 119 (47.9%) of v1 expert edges were preserved, and 62 were lost. That regression number is what convinced me to move off hand-tuned weights. *Learned:* hand weights feel principled until you measure them against a real label set. After that they feel arbitrary.

**v2. Multi-encoder stacker + PCA.** I added a second encoder, stacked it against the v1 signals, and ran PCA to compress the joint feature space before the classifier. Two-stage sacred evaluation exposed a majority-class collapse: the classifier was predicting 'Related' on almost every pair. The root cause was that PCA preserved the directions of largest variance but threw away the direction that separated the rare tiers. *Learned:* when the task is imbalanced class separation, PCA is not free dimensionality reduction; it can remove the signal you actually need.

**v3. Human-label domain adaptation.** I rewrote the feature pipeline around raw embeddings plus a random-forest calibrator trained on the newly labeled 150-pair human calibration split. Label shift correction was added to account for the fact that the test distribution has more unrelated pairs than the cal distribution. The pipeline stabilised around a 0.40 accuracy ceiling that I could not break through with text features alone. *Learned:* text-only features plateau on this task because many cross-framework pairs are conceptually related through structure that never appears verbatim in either description.

**v4. LLM-as-judge + cross-encoder ensemble.** Rather than keep squeezing more signal out of embeddings, I introduced a Claude Sonnet triple-vote judge: for each candidate pair I prompted Claude three times with different orderings and took the modal tier. I also added a cross-encoder ensemble (DeBERTa / ELECTRA / RoBERTa fine-tuned on the cal split) and graph features from a Node2Vec embedding. The RF calibrator consumed all of these. Frozen-test accuracy jumped to 0.468, the first real lift since v1. The ablation showed the LLM judge carried most of that lift; the cross-encoder ensemble added less than a percentage point. *Learned:* the LLM judge is the single most productive feature source on this task, and once it is in the stack the cross-encoder ensemble is expensive noise.

**v5. SBERT + LGBM stacker + SetFit + hierarchical LLM.** I replaced the RF calibrator with an LGBM stacker and added three more signal sources: SBERT sentence embeddings, a SetFit fine-tuned projection trained on the 150-pair cal split, and a hierarchical LLM prompt that asked Claude to first decide related-vs-unrelated and then decide the exact tier. Feature count hit 38. Frozen-test accuracy climbed to 0.505 but macro F1 flattened, and a mid-cycle experiment with a GAT + Mondrian-conformal variant collapsed to majority-class (tier accuracy 0.373) and tripped the break-glass gate. That was the moment I decided the pipeline was too wide. *Learned:* past a certain feature count, more features stops helping and starts hurting, and adding exotic components (GAT, SetFit, hierarchical prompts) made the pipeline harder to debug without proportional accuracy gain.

**v6. Reframed 22d GBM + split-conformal.** For v6 I threw out everything that did not earn its place: SBERT, SetFit, cross-encoder, LGBM stacker, PCA, hierarchical prompts, and the router features. The survivors are seven LLM features (the Sonnet triple-vote scores plus derived statistics), 13 structural features (graph depth, description length, Node2Vec cosine, GAT cosine, and four binary entry-type flags), and two Opus calibration features. 22 features total. The classifier is a single `GradientBoostingClassifier`; there is no stacker, no ensemble, no reranker. On top of that I wrapped the classifier in a split-conformal prediction procedure at α = 0.10 so every prediction ships with a calibrated set instead of a point estimate. Frozen tier accuracy is 0.5325, macro F1 is 0.489, adjacent accuracy is 0.84, and conformal coverage is 0.94, above the 0.90 nominal guarantee. *Learned:* pruning features is as important as adding them, and adding an honest uncertainty layer lets me ship a classifier whose mistakes I can at least predict.

**v7c. 3-model CE ensemble + GAT + LogReg.** The seventh iteration replaced the LLM triple-vote and Opus calibration features with cross-encoder soft probabilities from three transformer models (DeBERTa-v3-large, RoBERTa-large, DeBERTa-v3-base). Each model emits a 4-class probability vector per pair, yielding 12 CE features. Combined with 35 GAT features and 3 baseline features, the total feature count rose to 50. The classifier switched from GBM to regularized LogReg (C=0.01, selected by CV). The result was the largest accuracy jump in the series: 53.3% to 81.0% on the frozen test set. RoBERTa-large dominates feature importance, contributing the top four features by weight. Conformal coverage is 91.6% with an average set size of 1.56, a substantial improvement over v6's conformal behavior. The main remaining gap is the Equivalent tier (F1=0.0), limited by the 7-pair test sample.

> **Plain English:** Over seven rewrites the classifier went from a hand-tuned scoring formula to a 50-feature logistic regression powered by three cross-encoder transformer models. The biggest single lift came from swapping LLM-as-judge features for cross-encoder soft probabilities at v7c. The biggest single lesson came from v5 and v6, where adding noisy features stopped helping, and v7c worked better because it added high-quality features (transformer class probabilities) instead of noisy ones.

## Appendix B: Style guide

Every chart in this notebook follows a consistent visual language grounded in
four principles from the assigned readings.

**Color palettes.** Three palettes cover all data types in the corpus:

| Palette | Type | Use | Rationale |
|---------|------|-----|----------|
| `TIER_PALETTE` (4-color blue-teal ramp) | Sequential / ordinal | Expert-tier encoding throughout | Tiers are ordinal. Borner et al. (2019) and Borland & Taylor (2007) prescribe luminance-varying scales for ordered data. Lighter = weaker relationship, darker = stronger. |
| `FAMILY_COLOR` (3-color categorical) | Categorical | Feature-family encoding (GAT, Baseline, CE) | Three distinct hues for a nominal variable. Well under the 6-color ceiling recommended by Graze & Schwabish (2024). |
| `FRAMEWORK_PALETTE` (9-color Okabe-Ito) | Categorical | Framework identity in UMAP and composition charts | The Okabe-Ito palette is designed for colorblind accessibility (Okabe & Ito, 2002). Nine categories push the perceptual limit; the palette is augmented with one additional muted hue. |

**Heatmap colormaps.** Count-based heatmaps use `crest`, `Blues`, or `Purples`
(single-hue sequential ramps). The correlation matrix uses
`sns.diverging_palette(220, 20)`, a two-hue diverging ramp centered at zero,
following Borland & Taylor's (2007) recommendation that diverging palettes meet at a
perceptually neutral midpoint aligned with a meaningful data value. No rainbow or jet
colormaps appear anywhere in the notebook (Borland & Taylor, 2007).

**Perceptual encoding.** Bar charts, dot plots, and line charts dominate
because they map values to position along a common scale, the most accurate
perceptual channel (Cleveland & McGill, 1984). Stacked charts appear only
where the total is the primary message, with an explicit narrative note
acknowledging the perceptual trade-off. Area and angle encodings are avoided.

**Annotations.** On-plot annotations use a consistent style: black arrow
(`arrowstyle="->"`, `lw=1.0-1.2`), 9-10 pt text, placed to avoid data
overlap. Every annotation calls out a specific finding rather than restating
the axis labels.

**Typography.** `sns.set_theme(style="whitegrid", context="paper",
font_scale=1.15)` with bold 12 pt titles and 10 pt axis labels. The
whitegrid style provides reference lines without heavy chart borders,
reducing non-data ink (Tufte, 1983).